# Section 1

Initial model


In [1]:
# @title
import numpy as np
import ipywidgets as widgets
from IPython.display import display

def sge_model(
    DNA_sgRNA_per_million=0.375,  # µg per 1e6 cells (7.5 µg / 20 M)
    DNA_HDR_per_million=0.75,     # µg per 1e6 cells (15 µg / 20 M)
    cells_transfected=20e6,
    transfection_eff=0.6,
    editing_eff=0.4,
    library_size=2500,
    cells_retained=5e6,
    cells_pelleted=3e6,
    genomes_input=1.52e6,
    reads_per_replicate=10e6
):

    # Convert per-million-cell rates to total µg DNA
    DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
    DNA_HDR_total   = DNA_HDR_per_million * (cells_transfected / 1e6)

    # Effective edited cells
    edited_cells = cells_transfected * transfection_eff * editing_eff

    # Adjust for passage and pelleting
    edited_cells_retained = edited_cells * (cells_retained / cells_transfected)
    edited_cells_pelleted = edited_cells * (cells_pelleted / cells_transfected)

    # Fraction of edited genomes entering PCR
    edited_genomes_PCR = genomes_input * (edited_cells_pelleted / cells_pelleted)

    # Coverage metrics
    genomes_per_variant = genomes_input / library_size
    reads_per_variant   = reads_per_replicate / library_size
    reads_per_edited_genome = reads_per_variant / (genomes_per_variant * editing_eff)

    print("=== Transfection Inputs ===")
    print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
    print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
    print(f"HDR DNA: {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)")

    print("\n=== Editing Outcomes ===")
    print(f"Transfection efficiency: {transfection_eff*100:.0f}%")
    print(f"Editing efficiency: {editing_eff*100:.0f}%")
    print(f"Edited cells: {edited_cells/1e6:.2f} M")
    print(f"Edited cells retained: {edited_cells_retained/1e6:.2f} M")
    print(f"Edited genomes entering PCR: {edited_genomes_PCR/1e6:.2f} M")

    print("\n=== Coverage Metrics ===")
    print(f"Genomes per variant in PCR: {genomes_per_variant:.1f}")
    print(f"Reads per variant: {reads_per_variant:.1f}")
    print(f"Reads per edited genome: {reads_per_edited_genome:.2f}")

widgets.interact(
    sge_model,
    DNA_sgRNA_per_million=(0.1, 1.0, 0.05),
    DNA_HDR_per_million=(0.1, 1.5, 0.05),
    cells_transfected=(10e6, 30e6, 1e6),
    transfection_eff=(0.5, 0.7, 0.05),
    editing_eff=(0.25, 0.55, 0.05),
    library_size=(1000, 5000, 250),
    cells_retained=(2e6, 10e6, 1e6),
    cells_pelleted=(1e6, 5e6, 0.5e6),
    genomes_input=(1e6, 2e6, 0.1e6),
    reads_per_replicate=(5e6, 20e6, 1e6)
);


interactive(children=(FloatSlider(value=0.375, description='DNA_sgRNA_per_million', max=1.0, min=0.1, step=0.0…

# Section 2



*   Clarifies that Puro seletion occurs so all cells that survive selection have received guide + template

*   Sliders improved (format and labelling)



In [2]:
# @title
import numpy as np
import ipywidgets as widgets
from IPython.display import display

def sge_model(
    DNA_sgRNA_per_million=0.375,
    DNA_HDR_per_million=0.75,
    cells_transfected=20e6,
    transfection_eff=0.6,  # for reference only
    editing_eff=0.4,
    library_size=2500,
    cells_retained=5e6,
    cells_pelleted=3e6,
    genomes_input=1.52e6,
    reads_per_replicate=10e6
):
    # --- DNA inputs ---
    DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
    DNA_HDR_total   = DNA_HDR_per_million * (cells_transfected / 1e6)

    # --- Selection and editing ---
    selected_cells = cells_transfected  # all surviving are transfected after selection
    edited_cells = selected_cells * editing_eff

    # --- Passage and pelleting ---
    edited_cells_retained = edited_cells * (cells_retained / selected_cells)
    edited_cells_pelleted = edited_cells * (cells_pelleted / selected_cells)

    # --- PCR input ---
    edited_genomes_PCR = genomes_input * (edited_cells_pelleted / cells_pelleted)

    # --- Coverage metrics ---
    genomes_per_design = genomes_input / library_size
    edited_genomes_per_design = genomes_per_design * editing_eff
    reads_per_design = reads_per_replicate / library_size
    reads_per_edited_genome = reads_per_design / edited_genomes_per_design

    # --- Display results ---
    print("=== Transfection Inputs ===")
    print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
    print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
    print(f"HDR DNA: {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)")

    print("\n=== Post-Selection Outcomes ===")
    print(f"Transfection efficiency (for reference): {transfection_eff*100:.0f}%")
    print(f"Antibiotic selection: all surviving cells are transfected")
    print(f"Editing efficiency: {editing_eff*100:.0f}%")
    print(f"Edited cells: {edited_cells/1e6:.2f} M")
    print(f"Edited cells retained: {edited_cells_retained/1e6:.2f} M")
    print(f"Edited genomes entering PCR: {edited_genomes_PCR/1e6:.2f} M")

    print("\n=== Coverage Metrics ===")
    print(f"Total genomes per design (PCR input): {genomes_per_design:.1f}")
    print(f"Edited genomes per design: {edited_genomes_per_design:.1f}")
    print(f"Total reads per design: {reads_per_design:.1f}")
    print(f"Reads per edited genome: {reads_per_edited_genome:.2f}")

# --- Improved sliders with clear labels and scientific notation ---
style = {'description_width': '300px'}
layout = widgets.Layout(width='600px')

controls = {
    'DNA_sgRNA_per_million': widgets.FloatSlider(
        value=0.375, min=0.1, max=1.0, step=0.05,
        description='sgRNA DNA (µg per 10⁶ cells)',
        style=style, layout=layout, readout_format='.3f'
    ),
    'DNA_HDR_per_million': widgets.FloatSlider(
        value=0.75, min=0.1, max=1.5, step=0.05,
        description='HDR DNA (µg per 10⁶ cells)',
        style=style, layout=layout, readout_format='.3f'
    ),
    'cells_transfected': widgets.FloatSlider(
        value=20e6, min=10e6, max=30e6, step=1e6,
        description='Cells transfected',
        style=style, layout=layout, readout_format='.2e'
    ),
    'transfection_eff': widgets.FloatSlider(
        value=0.6, min=0.5, max=0.7, step=0.05,
        description='Transfection efficiency (reference only)',
        style=style, layout=layout, readout_format='.2f'
    ),
    'editing_eff': widgets.FloatSlider(
        value=0.4, min=0.25, max=0.55, step=0.05,
        description='Editing efficiency (fraction)',
        style=style, layout=layout, readout_format='.2f'
    ),
    'library_size': widgets.IntSlider(
        value=2500, min=1000, max=5000, step=250,
        description='Library size (design elements)',
        style=style, layout=layout
    ),
    'cells_retained': widgets.FloatSlider(
        value=5e6, min=2e6, max=10e6, step=1e6,
        description='Cells retained after passage',
        style=style, layout=layout, readout_format='.2e'
    ),
    'cells_pelleted': widgets.FloatSlider(
        value=3e6, min=1e6, max=5e6, step=0.5e6,
        description='Cells pelleted for gDNA',
        style=style, layout=layout, readout_format='.2e'
    ),
    'genomes_input': widgets.FloatSlider(
        value=1.52e6, min=1e6, max=2e6, step=0.1e6,
        description='Genomes input into PCR',
        style=style, layout=layout, readout_format='.2e'
    ),
    'reads_per_replicate': widgets.FloatSlider(
        value=10e6, min=5e6, max=20e6, step=1e6,
        description='Reads per replicate',
        style=style, layout=layout, readout_format='.2e'
    )
}

ui = widgets.VBox(list(controls.values()))
out = widgets.interactive_output(sge_model, controls)
display(ui, out)


Output()

# Section 3



*   Parameters can now be changed by either using the slider or inputting text into box. The 2 are linked so when you change one, the other changes to reflect this

*   Ability to change sgRNA and HDR input as the total per replicate, rather than just per million cells



In [3]:
# @title
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Core model function (unchanged except for using updated variable names) ---
def sge_model(
    DNA_sgRNA_per_million,
    DNA_HDR_per_million,
    cells_transfected,
    transfection_eff,
    editing_eff,
    library_size,
    cells_retained,
    cells_pelleted,
    genomes_input,
    reads_per_replicate
):
    DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
    DNA_HDR_total   = DNA_HDR_per_million * (cells_transfected / 1e6)

    selected_cells = cells_transfected
    edited_cells = selected_cells * editing_eff
    edited_cells_retained = edited_cells * (cells_retained / selected_cells)
    edited_cells_pelleted = edited_cells * (cells_pelleted / selected_cells)
    edited_genomes_PCR = genomes_input * (edited_cells_pelleted / cells_pelleted)

    genomes_per_design = genomes_input / library_size
    edited_genomes_per_design = genomes_per_design * editing_eff
    reads_per_design = reads_per_replicate / library_size
    reads_per_edited_genome = reads_per_design / edited_genomes_per_design

    clear_output(wait=True)
    print("=== Transfection Inputs ===")
    print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
    print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
    print(f"HDR DNA: {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)")

    print("\n=== Post-Selection Outcomes ===")
    print(f"Editing efficiency: {editing_eff*100:.0f}%")
    print(f"Edited cells retained: {edited_cells_retained/1e6:.2f} M")
    print(f"Edited genomes entering PCR: {edited_genomes_PCR/1e6:.2f} M")

    print("\n=== Coverage Metrics ===")
    print(f"Total genomes per design (PCR input): {genomes_per_design:.1f}")
    print(f"Edited genomes per design: {edited_genomes_per_design:.1f}")
    print(f"Total reads per design: {reads_per_design:.1f}")
    print(f"Reads per edited genome: {reads_per_edited_genome:.2f}")


# --- Helper to create linked slider + textbox ---
def linked_slider_text(value, minv, maxv, step, description, fmt='.3f', width='550px'):
    slider = widgets.FloatSlider(value=value, min=minv, max=maxv, step=step,
                                 description=description,
                                 style={'description_width': '280px'},
                                 layout=widgets.Layout(width=width),
                                 readout_format=fmt)
    text = widgets.FloatText(value=value, layout=widgets.Layout(width='120px'))
    widgets.jslink((slider, 'value'), (text, 'value'))
    return slider, text, widgets.HBox([slider, text])

# --- Core sliders ---
cells_transfected_s, cells_transfected_t, cells_transfected_box = linked_slider_text(20e6, 11e6, 50e6, 1e6, 'Cells transfected', fmt='.2e')
transfection_eff_s, transfection_eff_t, transfection_eff_box = linked_slider_text(0.6, 0.05, 0.9, 0.05, 'Transfection eff. (ref)', fmt='.2f')
editing_eff_s, editing_eff_t, editing_eff_box = linked_slider_text(0.4, 0.05, 0.9, 0.05, 'Editing efficiency', fmt='.2f')
library_size_s, library_size_t, library_size_box = linked_slider_text(2500, 500, 5000, 100, 'Library size', fmt='.0f')
cells_retained_s, cells_retained_t, cells_retained_box = linked_slider_text(5e6, 1e6, 10e6, 0.5e6, 'Cells retained', fmt='.2e')
cells_pelleted_s, cells_pelleted_t, cells_pelleted_box = linked_slider_text(3e6, 1e6, 10e6, 0.5e6, 'Cells pelleted', fmt='.2e')
genomes_input_s, genomes_input_t, genomes_input_box = linked_slider_text(1.52e6, 0.5e6, 5e6, 0.1e6, 'Genomes into PCR', fmt='.2e')
reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box = linked_slider_text(10e6, 1e6, 20e6, 1e6, 'Reads per replicate', fmt='.2e')

# --- Dual linked sgRNA / HDR controls ---
def make_dna_pair(label, default_per_million, minp, maxp, step):
    perM_slider, perM_text, perM_box = linked_slider_text(default_per_million, minp, maxp, step, f'{label} DNA (µg / 10⁶ cells)')
    total_text = widgets.FloatText(value=default_per_million * (cells_transfected_s.value / 1e6),
                                   description=f'{label} total (µg)',
                                   style={'description_width': '180px'},
                                   layout=widgets.Layout(width='300px'))
    # linkage: update total when perM or cells_transfected changes
    def update_total(*args):
        total_text.value = perM_slider.value * (cells_transfected_s.value / 1e6)
    perM_slider.observe(update_total, 'value')
    cells_transfected_s.observe(update_total, 'value')
    # reverse linkage: update perM when total changes
    def update_perM(change):
        perM_slider.value = total_text.value / (cells_transfected_s.value / 1e6)
    total_text.observe(update_perM, 'value')
    return perM_slider, perM_text, total_text, widgets.HBox([perM_slider, perM_text, total_text])

sgRNA_s, sgRNA_t, sgRNA_total_t, sgRNA_box = make_dna_pair('sgRNA', 0.375, 0.1, 1.0, 0.05)
HDR_s, HDR_t, HDR_total_t, HDR_box = make_dna_pair('HDR', 0.75, 0.1, 1.5, 0.05)

# --- Group layout ---
ui = widgets.VBox([
    widgets.HTML("<h3>DNA Inputs</h3>"),
    sgRNA_box,
    HDR_box,
    widgets.HTML("<h3>Cell and Editing Parameters</h3>"),
    cells_transfected_box,
    transfection_eff_box,
    editing_eff_box,
    library_size_box,
    cells_retained_box,
    cells_pelleted_box,
    genomes_input_box,
    reads_per_replicate_box
])

out = widgets.interactive_output(
    sge_model,
    {
        'DNA_sgRNA_per_million': sgRNA_s,
        'DNA_HDR_per_million': HDR_s,
        'cells_transfected': cells_transfected_s,
        'transfection_eff': transfection_eff_s,
        'editing_eff': editing_eff_s,
        'library_size': library_size_s,
        'cells_retained': cells_retained_s,
        'cells_pelleted': cells_pelleted_s,
        'genomes_input': genomes_input_s,
        'reads_per_replicate': reads_per_replicate_s
    }
)

display(ui, out)


Output()

# Section 4


*   Addition of calculation for how much DNA is needed for full screen (all replicates + error)
*   Addition of ability to interactively manipulate HDR:sgRNA ratio



In [4]:
# @title
# =============================================
#      SGE SCREEN TOY MODEL — INTERACTIVE UI
# =============================================
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------------------------------
# Core model computation
# ------------------------------------------------------
def sge_model(
    DNA_sgRNA_per_million,
    DNA_HDR_per_million,
    cells_transfected,
    transfection_eff,
    editing_eff,
    library_size,
    cells_retained,
    cells_pelleted,
    genomes_input,
    reads_per_replicate
):
    # Compute totals for one transfection
    DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
    DNA_HDR_total   = DNA_HDR_per_million * (cells_transfected / 1e6)

    # Apply full-screen multiplier (3 replicates + 10% buffer + 0.1 replicate)
    full_screen_multiplier = 3.1 * 1.1
    DNA_sgRNA_full = DNA_sgRNA_total * full_screen_multiplier
    DNA_HDR_full   = DNA_HDR_total * full_screen_multiplier

    # Model editing & selection
    selected_cells = cells_transfected  # all survive due to antibiotic marker
    edited_cells = selected_cells * editing_eff
    edited_cells_retained = edited_cells * (cells_retained / selected_cells)
    edited_cells_pelleted = edited_cells * (cells_pelleted / selected_cells)
    edited_genomes_PCR = genomes_input * (edited_cells_pelleted / cells_pelleted)

    # Coverage metrics
    genomes_per_design = genomes_input / library_size
    edited_genomes_per_design = genomes_per_design * editing_eff
    reads_per_design = reads_per_replicate / library_size
    reads_per_edited_genome = reads_per_design / edited_genomes_per_design

    clear_output(wait=True)
    print("=== Transfection Inputs ===")
    print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
    print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
    print(f"HDR DNA: {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)")
    print(f"\nFull screen requirement (3× reps + 0.1 margin + 10% buffer): ×3.41")
    print(f"→ sgRNA total required: {DNA_sgRNA_full:.2f} µg")
    print(f"→ HDR total required:   {DNA_HDR_full:.2f} µg")

    print("\n=== Post-Selection Outcomes ===")
    print(f"Editing efficiency: {editing_eff*100:.0f}%")
    print(f"Edited cells retained: {edited_cells_retained/1e6:.2f} M")
    print(f"Edited genomes entering PCR: {edited_genomes_PCR/1e6:.2f} M")

    print("\n=== Coverage Metrics ===")
    print(f"Total genomes per design (PCR input): {genomes_per_design:.1f}")
    print(f"Edited genomes per design: {edited_genomes_per_design:.1f}")
    print(f"Total reads per design: {reads_per_design:.1f}")
    print(f"Reads per edited genome: {reads_per_edited_genome:.2f}")

# ------------------------------------------------------
# Helper: slider + textbox pair (two-way linked)
# ------------------------------------------------------
def linked_slider_text(value, minv, maxv, step, description, fmt='.3f', width='550px'):
    slider = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description,
        style={'description_width': '280px'},
        layout=widgets.Layout(width=width),
        readout_format=fmt
    )
    text = widgets.FloatText(value=value, layout=widgets.Layout(width='120px'))
    widgets.jslink((slider, 'value'), (text, 'value'))
    return slider, text, widgets.HBox([slider, text])

# ------------------------------------------------------
# Core sliders (boundaries set here)
# ------------------------------------------------------
cells_transfected_s, cells_transfected_t, cells_transfected_box = linked_slider_text(20e6, 11e6, 50e6, 1e6, 'Cells transfected', fmt='.2e')
transfection_eff_s, transfection_eff_t, transfection_eff_box = linked_slider_text(0.6, 0.05, 0.9, 0.05, 'Transfection eff. (ref)', fmt='.2f')
editing_eff_s, editing_eff_t, editing_eff_box = linked_slider_text(0.4, 0.05, 0.9, 0.05, 'Editing efficiency', fmt='.2f')
library_size_s, library_size_t, library_size_box = linked_slider_text(2500, 500, 5000, 100, 'Library size', fmt='.0f')
cells_retained_s, cells_retained_t, cells_retained_box = linked_slider_text(5e6, 1e6, 10e6, 0.5e6, 'Cells retained', fmt='.2e')
cells_pelleted_s, cells_pelleted_t, cells_pelleted_box = linked_slider_text(3e6, 1e6, 10e6, 0.5e6, 'Cells pelleted', fmt='.2e')
genomes_input_s, genomes_input_t, genomes_input_box = linked_slider_text(1.52e6, 0.5e6, 5e6, 0.1e6, 'Genomes into PCR', fmt='.2e')
reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box = linked_slider_text(10e6, 1e6, 20e6, 1e6, 'Reads per replicate', fmt='.2e')

# ------------------------------------------------------
# DNA sliders (per-million ↔ total linked)
# ------------------------------------------------------
def make_dna_pair(label, default_per_million, minp, maxp, step):
    perM_slider, perM_text, perM_box = linked_slider_text(
        default_per_million, minp, maxp, step, f'{label} DNA (µg / 10⁶ cells)'
    )
    total_text = widgets.FloatText(
        value=default_per_million * (cells_transfected_s.value / 1e6),
        description=f'{label} total (µg)',
        style={'description_width': '180px'},
        layout=widgets.Layout(width='300px')
    )

    def update_total(*args):
        total_text.value = perM_slider.value * (cells_transfected_s.value / 1e6)
    perM_slider.observe(update_total, 'value')
    cells_transfected_s.observe(update_total, 'value')

    def update_perM(change):
        perM_slider.value = total_text.value / (cells_transfected_s.value / 1e6)
    total_text.observe(update_perM, 'value')

    return perM_slider, perM_text, total_text, widgets.HBox([perM_slider, perM_text, total_text])

sgRNA_s, sgRNA_t, sgRNA_total_t, sgRNA_box = make_dna_pair('sgRNA', 0.375, 0.1, 1.0, 0.05)
HDR_s, HDR_t, HDR_total_t, HDR_box = make_dna_pair('HDR', 0.75, 0.1, 1.5, 0.05)

# ------------------------------------------------------
# HDR : sgRNA ratio linking
# ------------------------------------------------------
ratio_slider, ratio_text, ratio_box = linked_slider_text(
    value=HDR_s.value / sgRNA_s.value,
    minv=0.5, maxv=4.0, step=0.05,
    description='HDR : sgRNA ratio',
    fmt='.2f'
)

_updating = {'ratio': False, 'sgRNA': False, 'HDR': False}

def update_ratio(*args):
    if _updating['ratio']: return
    _updating['ratio'] = True
    ratio_slider.value = HDR_s.value / sgRNA_s.value
    ratio_text.value = HDR_s.value / sgRNA_s.value
    _updating['ratio'] = False

def update_HDR_from_ratio(*args):
    if _updating['HDR']: return
    _updating['HDR'] = True
    HDR_s.value = sgRNA_s.value * ratio_slider.value
    _updating['HDR'] = False
    update_ratio()

def update_from_HDR(*args):
    if _updating['sgRNA']: return
    _updating['sgRNA'] = True
    ratio_slider.value = HDR_s.value / sgRNA_s.value
    ratio_text.value = HDR_s.value / sgRNA_s.value
    _updating['sgRNA'] = False
    update_ratio()

sgRNA_s.observe(update_HDR_from_ratio, 'value')
HDR_s.observe(update_from_HDR, 'value')
ratio_slider.observe(update_HDR_from_ratio, 'value')
ratio_text.observe(update_HDR_from_ratio, 'value')

# ------------------------------------------------------
# Layout and interactive output
# ------------------------------------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>DNA Inputs</h3>"),
    sgRNA_box,
    HDR_box,
    ratio_box,
    widgets.HTML("<h3>Cell and Editing Parameters</h3>"),
    cells_transfected_box,
    transfection_eff_box,
    editing_eff_box,
    library_size_box,
    cells_retained_box,
    cells_pelleted_box,
    genomes_input_box,
    reads_per_replicate_box
])

out = widgets.interactive_output(
    sge_model,
    {
        'DNA_sgRNA_per_million': sgRNA_s,
        'DNA_HDR_per_million': HDR_s,
        'cells_transfected': cells_transfected_s,
        'transfection_eff': transfection_eff_s,
        'editing_eff': editing_eff_s,
        'library_size': library_size_s,
        'cells_retained': cells_retained_s,
        'cells_pelleted': cells_pelleted_s,
        'genomes_input': genomes_input_s,
        'reads_per_replicate': reads_per_replicate_s
    }
)

display(ui, out)


Output()

# Section 5


*   Matplotlib plot visualises coverage across pipeline stages




In [5]:
# @title
# =============================================
#     SGE SCREEN TOY MODEL — FULL INTERACTIVE UI
# =============================================
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# ------------------------------------------------------
# Helper: slider + textbox pair (two-way linked)
# ------------------------------------------------------
def linked_slider_text(value, minv, maxv, step, description, fmt='.3f', width='550px'):
    slider = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description,
        style={'description_width': '280px'},
        layout=widgets.Layout(width=width),
        readout_format=fmt
    )
    text = widgets.FloatText(value=value, layout=widgets.Layout(width='120px'))
    widgets.jslink((slider, 'value'), (text, 'value'))
    return slider, text, widgets.HBox([slider, text])

# ------------------------------------------------------
# Core sliders
# ------------------------------------------------------
cells_transfected_s, cells_transfected_t, cells_transfected_box = linked_slider_text(20e6, 11e6, 50e6, 1e6, 'Cells transfected', fmt='.2e')
transfection_eff_s, transfection_eff_t, transfection_eff_box = linked_slider_text(0.6, 0.05, 0.9, 0.05, 'Transfection eff. (ref)', fmt='.2f')
editing_eff_s, editing_eff_t, editing_eff_box = linked_slider_text(0.4, 0.05, 0.9, 0.05, 'Editing efficiency', fmt='.2f')
library_size_s, library_size_t, library_size_box = linked_slider_text(2500, 500, 5000, 100, 'Library size', fmt='.0f')
cells_retained_s, cells_retained_t, cells_retained_box = linked_slider_text(5e6, 1e6, 10e6, 0.5e6, 'Cells retained', fmt='.2e')
cells_pelleted_s, cells_pelleted_t, cells_pelleted_box = linked_slider_text(3e6, 1e6, 10e6, 0.5e6, 'Cells pelleted', fmt='.2e')
genomes_input_s, genomes_input_t, genomes_input_box = linked_slider_text(1.52e6, 0.5e6, 5e6, 0.1e6, 'Genomes into PCR', fmt='.2e')
reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box = linked_slider_text(10e6, 1e6, 20e6, 1e6, 'Reads per replicate', fmt='.2e')

# ------------------------------------------------------
# DNA sliders (per-million ↔ total linked)
# ------------------------------------------------------
def make_dna_pair(label, default_per_million, minp, maxp, step):
    perM_slider, perM_text, perM_box = linked_slider_text(
        default_per_million, minp, maxp, step, f'{label} DNA (µg / 10⁶ cells)'
    )
    total_text = widgets.FloatText(
        value=default_per_million * (cells_transfected_s.value / 1e6),
        description=f'{label} total (µg)',
        style={'description_width': '180px'},
        layout=widgets.Layout(width='300px')
    )

    def update_total(*args):
        total_text.value = perM_slider.value * (cells_transfected_s.value / 1e6)
    perM_slider.observe(update_total, 'value')
    cells_transfected_s.observe(update_total, 'value')

    def update_perM(change):
        perM_slider.value = total_text.value / (cells_transfected_s.value / 1e6)
    total_text.observe(update_perM, 'value')

    return perM_slider, perM_text, total_text, widgets.HBox([perM_slider, perM_text, total_text])

sgRNA_s, sgRNA_t, sgRNA_total_t, sgRNA_box = make_dna_pair('sgRNA', 0.375, 0.1, 1.0, 0.05)
HDR_s, HDR_t, HDR_total_t, HDR_box = make_dna_pair('HDR', 0.75, 0.1, 1.5, 0.05)

# ------------------------------------------------------
# HDR : sgRNA ratio linking
# ------------------------------------------------------
ratio_slider, ratio_text, ratio_box = linked_slider_text(
    value=HDR_s.value / sgRNA_s.value,
    minv=0.5, maxv=4.0, step=0.05,
    description='HDR : sgRNA ratio',
    fmt='.2f'
)

_updating = {'ratio': False, 'sgRNA': False, 'HDR': False}

def update_ratio(*args):
    if _updating['ratio']: return
    _updating['ratio'] = True
    ratio_slider.value = HDR_s.value / sgRNA_s.value
    ratio_text.value = HDR_s.value / sgRNA_s.value
    _updating['ratio'] = False

def update_HDR_from_ratio(*args):
    if _updating['HDR']: return
    _updating['HDR'] = True
    HDR_s.value = sgRNA_s.value * ratio_slider.value
    _updating['HDR'] = False
    update_ratio()

def update_from_HDR(*args):
    if _updating['sgRNA']: return
    _updating['sgRNA'] = True
    ratio_slider.value = HDR_s.value / sgRNA_s.value
    ratio_text.value = HDR_s.value / sgRNA_s.value
    _updating['sgRNA'] = False
    update_ratio()

sgRNA_s.observe(update_HDR_from_ratio, 'value')
HDR_s.observe(update_from_HDR, 'value')
ratio_slider.observe(update_HDR_from_ratio, 'value')
ratio_text.observe(update_HDR_from_ratio, 'value')

# ------------------------------------------------------
# Output area
# ------------------------------------------------------
output_text = widgets.Output()

# ------------------------------------------------------
# Model computation & plotting
# ------------------------------------------------------
def sge_model(
    DNA_sgRNA_per_million,
    DNA_HDR_per_million,
    cells_transfected,
    transfection_eff,
    editing_eff,
    library_size,
    cells_retained,
    cells_pelleted,
    genomes_input,
    reads_per_replicate
):
    with output_text:
        clear_output(wait=True)

        # DNA totals
        DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
        DNA_HDR_total   = DNA_HDR_per_million * (cells_transfected / 1e6)

        full_screen_multiplier = 3.1 * 1.1
        DNA_sgRNA_full = DNA_sgRNA_total * full_screen_multiplier
        DNA_HDR_full   = DNA_HDR_total * full_screen_multiplier

        # Model biology
        selected_cells = cells_transfected
        edited_cells = selected_cells * editing_eff
        edited_cells_retained = edited_cells * (cells_retained / selected_cells)
        edited_genomes_PCR = genomes_input * (edited_cells_retained / cells_pelleted)

        # Coverage metrics
        coverage = {
            "Transfected": cells_transfected / library_size,
            "Edited": edited_cells / library_size,
            "PCR Input": genomes_input / library_size,
            "Reads": reads_per_replicate / library_size
        }

        # Text summary
        print("=== Transfection Inputs ===")
        print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
        print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
        print(f"HDR DNA:   {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)\n")

        print(f"Full screen requirement (3× reps + 0.1 margin + 10% buffer): ×{full_screen_multiplier:.2f}")
        print(f"→ sgRNA total required: {DNA_sgRNA_full:.2f} µg")
        print(f"→ HDR total required:   {DNA_HDR_full:.2f} µg\n")

        print("=== Post-Selection Outcomes ===")
        print(f"Editing efficiency: {editing_eff*100:.0f}%")
        print(f"Edited cells retained: {edited_cells_retained/1e6:.2f} M")
        print(f"Edited genomes entering PCR: {edited_genomes_PCR/1e6:.2f} M\n")

        print("=== Coverage Metrics ===")
        print(f"Total genomes per design (PCR input): {coverage['PCR Input']:.1f}")
        print(f"Edited genomes per design: {coverage['Edited']:.1f}")
        print(f"Total reads per design: {coverage['Reads']:.1f}")
        print(f"Reads per edited genome: {coverage['Reads']/coverage['Edited']:.2f}\n")

        # Plot coverage
        stages = list(coverage.keys())
        values = list(coverage.values())
        plt.figure(figsize=(6,4))
        plt.plot(stages, values, marker='o')
        plt.yscale('log')
        plt.ylabel('Coverage per variant (log scale)')
        plt.title('Coverage Across Pipeline Stages')
        plt.grid(True, which="both", ls="--", lw=0.5)
        plt.show()

# ------------------------------------------------------
# Layout
# ------------------------------------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>DNA Inputs</h3>"),
    sgRNA_box,
    HDR_box,
    ratio_box,
    widgets.HTML("<h3>Cell and Editing Parameters</h3>"),
    cells_transfected_box,
    transfection_eff_box,
    editing_eff_box,
    library_size_box,
    cells_retained_box,
    cells_pelleted_box,
    genomes_input_box,
    reads_per_replicate_box
])

out = widgets.interactive_output(
    sge_model,
    {
        'DNA_sgRNA_per_million': sgRNA_s,
        'DNA_HDR_per_million': HDR_s,
        'cells_transfected': cells_transfected_s,
        'transfection_eff': transfection_eff_s,
        'editing_eff': editing_eff_s,
        'library_size': library_size_s,
        'cells_retained': cells_retained_s,
        'cells_pelleted': cells_pelleted_s,
        'genomes_input': genomes_input_s,
        'reads_per_replicate': reads_per_replicate_s
    }
)

display(ui, out, output_text)


Output()

Output()

# Section 6



*   Improved plotting




In [6]:
# @title
# =============================================
#     SGE SCREEN TOY MODEL — FULL INTERACTIVE UI
# =============================================
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# ------------------------------------------------------
# Helper: slider + textbox pair (two-way linked)
# ------------------------------------------------------
def linked_slider_text(value, minv, maxv, step, description, fmt='.3f', width='550px'):
    slider = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description,
        style={'description_width': '280px'},
        layout=widgets.Layout(width=width),
        readout_format=fmt
    )
    text = widgets.FloatText(value=value, layout=widgets.Layout(width='120px'))
    widgets.jslink((slider, 'value'), (text, 'value'))
    return slider, text, widgets.HBox([slider, text])

# ------------------------------------------------------
# Core sliders
# ------------------------------------------------------
cells_transfected_s, cells_transfected_t, cells_transfected_box = linked_slider_text(20e6, 11e6, 50e6, 1e6, 'Cells transfected', fmt='.2e')
transfection_eff_s, transfection_eff_t, transfection_eff_box = linked_slider_text(0.6, 0.05, 0.9, 0.05, 'Transfection eff. (ref)', fmt='.2f')
editing_eff_s, editing_eff_t, editing_eff_box = linked_slider_text(0.4, 0.05, 0.9, 0.05, 'Editing efficiency', fmt='.2f')
library_size_s, library_size_t, library_size_box = linked_slider_text(2500, 500, 5000, 100, 'Library size', fmt='.0f')
cells_retained_s, cells_retained_t, cells_retained_box = linked_slider_text(5e6, 1e6, 10e6, 0.5e6, 'Cells retained', fmt='.2e')
cells_pelleted_s, cells_pelleted_t, cells_pelleted_box = linked_slider_text(3e6, 1e6, 10e6, 0.5e6, 'Cells pelleted', fmt='.2e')
genomes_input_s, genomes_input_t, genomes_input_box = linked_slider_text(1.52e6, 0.5e6, 5e6, 0.1e6, 'Genomes into PCR', fmt='.2e')
reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box = linked_slider_text(10e6, 1e6, 20e6, 1e6, 'Reads per replicate', fmt='.2e')

# ------------------------------------------------------
# DNA sliders (per-million ↔ total linked)
# ------------------------------------------------------
def make_dna_pair(label, default_per_million, minp, maxp, step):
    perM_slider, perM_text, perM_box = linked_slider_text(
        default_per_million, minp, maxp, step, f'{label} DNA (µg / 10⁶ cells)'
    )
    total_text = widgets.FloatText(
        value=default_per_million * (cells_transfected_s.value / 1e6),
        description=f'{label} total (µg)',
        style={'description_width': '180px'},
        layout=widgets.Layout(width='300px')
    )

    def update_total(*args):
        total_text.value = perM_slider.value * (cells_transfected_s.value / 1e6)
    perM_slider.observe(update_total, 'value')
    cells_transfected_s.observe(update_total, 'value')

    def update_perM(change):
        perM_slider.value = total_text.value / (cells_transfected_s.value / 1e6)
    total_text.observe(update_perM, 'value')

    return perM_slider, perM_text, total_text, widgets.HBox([perM_slider, perM_text, total_text])

sgRNA_s, sgRNA_t, sgRNA_total_t, sgRNA_box = make_dna_pair('sgRNA', 0.375, 0.1, 1.0, 0.05)
HDR_s, HDR_t, HDR_total_t, HDR_box = make_dna_pair('HDR', 0.75, 0.1, 1.5, 0.05)

# ------------------------------------------------------
# HDR : sgRNA ratio linking
# ------------------------------------------------------
ratio_slider, ratio_text, ratio_box = linked_slider_text(
    value=HDR_s.value / sgRNA_s.value,
    minv=0.5, maxv=4.0, step=0.05,
    description='HDR : sgRNA ratio',
    fmt='.2f'
)

_updating = {'ratio': False, 'sgRNA': False, 'HDR': False}

def update_ratio(*args):
    if _updating['ratio']: return
    _updating['ratio'] = True
    ratio_slider.value = HDR_s.value / sgRNA_s.value
    ratio_text.value = HDR_s.value / sgRNA_s.value
    _updating['ratio'] = False

def update_HDR_from_ratio(*args):
    if _updating['HDR']: return
    _updating['HDR'] = True
    HDR_s.value = sgRNA_s.value * ratio_slider.value
    _updating['HDR'] = False
    update_ratio()

def update_from_HDR(*args):
    if _updating['sgRNA']: return
    _updating['sgRNA'] = True
    ratio_slider.value = HDR_s.value / sgRNA_s.value
    ratio_text.value = HDR_s.value / sgRNA_s.value
    _updating['sgRNA'] = False
    update_ratio()

sgRNA_s.observe(update_HDR_from_ratio, 'value')
HDR_s.observe(update_from_HDR, 'value')
ratio_slider.observe(update_HDR_from_ratio, 'value')
ratio_text.observe(update_HDR_from_ratio, 'value')

# ------------------------------------------------------
# Output area
# ------------------------------------------------------
output_text = widgets.Output()

# ------------------------------------------------------
# Model computation & plotting
# ------------------------------------------------------
def sge_model(
    DNA_sgRNA_per_million,
    DNA_HDR_per_million,
    cells_transfected,
    transfection_eff,
    editing_eff,
    library_size,
    cells_retained,
    cells_pelleted,
    genomes_input,
    reads_per_replicate
):
    with output_text:
        clear_output(wait=True)

        # DNA totals
        DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
        DNA_HDR_total   = DNA_HDR_per_million * (cells_transfected / 1e6)

        full_screen_multiplier = 3.1 * 1.1
        DNA_sgRNA_full = DNA_sgRNA_total * full_screen_multiplier
        DNA_HDR_full   = DNA_HDR_total * full_screen_multiplier

        # Model biology
        selected_cells = cells_transfected
        edited_cells = selected_cells * editing_eff
        edited_cells_retained = edited_cells * (cells_retained / selected_cells)
        edited_genomes_PCR = genomes_input * (edited_cells_retained / cells_pelleted)

        # Coverage metrics (all per variant)
        coverage = {
            "Transfected": cells_transfected / library_size,
            "Edited": edited_cells / library_size,
            "Edited Retained": edited_cells_retained / library_size,
            "PCR Input": genomes_input / library_size,
            "Reads": reads_per_replicate / library_size
        }

        # Text summary
        print("=== Transfection Inputs ===")
        print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
        print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
        print(f"HDR DNA:   {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)\n")

        print(f"Full screen requirement (3× reps + 0.1 margin + 10% buffer): ×{full_screen_multiplier:.2f}")
        print(f"→ sgRNA total required: {DNA_sgRNA_full:.2f} µg")
        print(f"→ HDR total required:   {DNA_HDR_full:.2f} µg\n")

        print("=== Post-Selection Outcomes ===")
        print(f"Editing efficiency: {editing_eff*100:.0f}%")
        print(f"Edited cells retained: {edited_cells_retained/1e6:.2f} M")
        print(f"Edited genomes entering PCR: {edited_genomes_PCR/1e6:.2f} M\n")

        print("=== Coverage Metrics ===")
        print(f"Total genomes per design (PCR input): {coverage['PCR Input']:.1f}")
        print(f"Edited genomes per design: {coverage['Edited']:.1f}")
        print(f"Edited retained per design: {coverage['Edited Retained']:.1f}")
        print(f"Total reads per design: {coverage['Reads']:.1f}")
        print(f"Reads per edited genome: {coverage['Reads']/coverage['Edited']:.2f}\n")

        # --- Plot coverage graph ---
        stages = list(coverage.keys())
        values = list(coverage.values())

        plt.figure(figsize=(7,4.5))
        plt.plot(stages, values, marker='o', lw=2)
        plt.yscale('log')
        plt.ylabel('Coverage per variant (log scale)')
        plt.title('Coverage Across Pipeline Stages')
        plt.grid(True, which="both", ls="--", lw=0.5)

        # Annotate each point with raw value
        for i, (x, y) in enumerate(zip(stages, values)):
            plt.text(i, y*1.05, f"{y:.1f}", ha='center', va='bottom', fontsize=9)

        plt.show()

# ------------------------------------------------------
# Layout
# ------------------------------------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>DNA Inputs</h3>"),
    sgRNA_box,
    HDR_box,
    ratio_box,
    widgets.HTML("<h3>Cell and Editing Parameters</h3>"),
    cells_transfected_box,
    transfection_eff_box,
    editing_eff_box,
    library_size_box,
    cells_retained_box,
    cells_pelleted_box,
    genomes_input_box,
    reads_per_replicate_box
])

out = widgets.interactive_output(
    sge_model,
    {
        'DNA_sgRNA_per_million': sgRNA_s,
        'DNA_HDR_per_million': HDR_s,
        'cells_transfected': cells_transfected_s,
        'transfection_eff': transfection_eff_s,
        'editing_eff': editing_eff_s,
        'library_size': library_size_s,
        'cells_retained': cells_retained_s,
        'cells_pelleted': cells_pelleted_s,
        'genomes_input': genomes_input_s,
        'reads_per_replicate': reads_per_replicate_s
    }
)

display(ui, out, output_text)


Output()

Output()

# Section 7


*   Advanced visualisations added:
      * Sweep plot: Choose a single parameter and visualise how other parameters are affected when the inital parameter is varied
      * Compare scenarios: Choose the parameters for 3 scenarios and compare how that leads to changes in coverage metrics






In [7]:
# @title
# =============================================
#    SGE SCREEN MODEL — INTERACTIVE (Plotly)
# =============================================
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------
# Helper: slider + textbox pair
# ------------------------------
def linked_slider_text(value, minv, maxv, step, description, fmt='.3f', width='560px'):
    slider = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description,
        style={'description_width': '300px'},
        layout=widgets.Layout(width=width),
        readout_format=fmt
    )
    text = widgets.FloatText(value=value, layout=widgets.Layout(width='120px'))
    widgets.jslink((slider, 'value'), (text, 'value'))
    return slider, text, widgets.HBox([slider, text])

# ------------------------------
# Core sliders (tune boundaries here)
# ------------------------------
cells_transfected_s, cells_transfected_t, cells_transfected_box = linked_slider_text(20e6, 11e6, 50e6, 1e6, 'Cells transfected', fmt='.2e')
transfection_eff_s, transfection_eff_t, transfection_eff_box = linked_slider_text(0.6, 0.05, 0.9, 0.05, 'Transfection eff. (ref)', fmt='.2f')
editing_eff_s, editing_eff_t, editing_eff_box = linked_slider_text(0.4, 0.05, 0.9, 0.05, 'Editing efficiency', fmt='.2f')
library_size_s, library_size_t, library_size_box = linked_slider_text(2500, 500, 5000, 100, 'Library size', fmt='.0f')
cells_retained_s, cells_retained_t, cells_retained_box = linked_slider_text(5e6, 1e6, 10e6, 0.5e6, 'Cells retained', fmt='.2e')
cells_pelleted_s, cells_pelleted_t, cells_pelleted_box = linked_slider_text(3e6, 1e6, 10e6, 0.5e6, 'Cells pelleted', fmt='.2e')
genomes_input_s, genomes_input_t, genomes_input_box = linked_slider_text(1.52e6, 0.5e6, 5e6, 0.1e6, 'Genomes into PCR', fmt='.2e')
reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box = linked_slider_text(10e6, 1e6, 20e6, 1e6, 'Reads per replicate', fmt='.2e')

# ------------------------------
# DNA sliders (per-million ↔ total linked)
# ------------------------------
def make_dna_pair(label, default_per_million, minp, maxp, step):
    perM_slider, perM_text, perM_box = linked_slider_text(default_per_million, minp, maxp, step, f'{label} DNA (µg / 10⁶ cells)')
    total_text = widgets.FloatText(
        value=default_per_million * (cells_transfected_s.value / 1e6),
        description=f'{label} total (µg)',
        style={'description_width': '180px'},
        layout=widgets.Layout(width='320px')
    )

    def update_total(*args):
        total_text.value = perM_slider.value * (cells_transfected_s.value / 1e6)
    perM_slider.observe(update_total, 'value')
    cells_transfected_s.observe(update_total, 'value')

    def update_perM(change):
        # avoid division by zero
        denom = (cells_transfected_s.value / 1e6) if cells_transfected_s.value != 0 else 1e-6
        perM_slider.value = total_text.value / denom
    total_text.observe(update_perM, 'value')

    return perM_slider, perM_text, total_text, widgets.HBox([perM_slider, perM_text, total_text])

sgRNA_s, sgRNA_t, sgRNA_total_t, sgRNA_box = make_dna_pair('sgRNA', 0.375, 0.1, 1.0, 0.05)
HDR_s, HDR_t, HDR_total_t, HDR_box = make_dna_pair('HDR', 0.75, 0.1, 1.5, 0.05)

# ------------------------------
# HDR : sgRNA ratio linking
# ------------------------------
ratio_slider, ratio_text, ratio_box = linked_slider_text(
    value=HDR_s.value / sgRNA_s.value,
    minv=0.5, maxv=4.0, step=0.05,
    description='HDR : sgRNA ratio',
    fmt='.2f'
)

_updating = {'ratio': False, 'sgRNA': False, 'HDR': False}

def update_ratio(*args):
    if _updating['ratio']: return
    _updating['ratio'] = True
    # guard division by zero
    if sgRNA_s.value == 0:
        ratio = np.nan
    else:
        ratio = HDR_s.value / sgRNA_s.value
    ratio_slider.value = ratio
    ratio_text.value = ratio
    _updating['ratio'] = False

def update_HDR_from_ratio(*args):
    if _updating['HDR']: return
    _updating['HDR'] = True
    HDR_s.value = sgRNA_s.value * ratio_slider.value
    _updating['HDR'] = False
    update_ratio()

def update_from_HDR(*args):
    if _updating['sgRNA']: return
    _updating['sgRNA'] = True
    # update ratio based on new HDR value
    if sgRNA_s.value == 0:
        r = np.nan
    else:
        r = HDR_s.value / sgRNA_s.value
    ratio_slider.value = r
    ratio_text.value = r
    _updating['sgRNA'] = False
    update_ratio()

sgRNA_s.observe(update_HDR_from_ratio, 'value')
HDR_s.observe(update_from_HDR, 'value')
ratio_slider.observe(update_HDR_from_ratio, 'value')
ratio_text.observe(update_HDR_from_ratio, 'value')

# ------------------------------
# Live labels for full-screen DNA requirements
# ------------------------------
sgRNA_full_label = widgets.HTML()
HDR_full_label = widgets.HTML()

def update_full_screen_labels():
    per_run_sg = sgRNA_s.value * (cells_transfected_s.value / 1e6)
    per_run_hdr = HDR_s.value * (cells_transfected_s.value / 1e6)
    mult = 3.1 * 1.1
    sg_full = per_run_sg * mult
    hdr_full = per_run_hdr * mult
    sgRNA_full_label.value = f"<b>sgRNA per run:</b> {per_run_sg:.2f} µg &nbsp;&nbsp; <b>full screen (×{mult:.2f}):</b> {sg_full:.2f} µg"
    HDR_full_label.value = f"<b>HDR per run:</b> {per_run_hdr:.2f} µg &nbsp;&nbsp; <b>full screen (×{mult:.2f}):</b> {hdr_full:.2f} µg"

# attach updater observers
sgRNA_s.observe(lambda *_: update_full_screen_labels(), 'value')
HDR_s.observe(lambda *_: update_full_screen_labels(), 'value')
cells_transfected_s.observe(lambda *_: update_full_screen_labels(), 'value')
update_full_screen_labels()

# ------------------------------
# Output areas
# ------------------------------
output_text = widgets.Output()
coverage_plot_out = widgets.Output()

# ------------------------------
# Core model computation & display (text + matplotlib coverage)
# ------------------------------
def compute_metrics(cells_transfected, editing_eff, library_size, cells_retained, cells_pelleted, genomes_input, reads_per_replicate):
    # quantities (absolute)
    selected_cells = cells_transfected
    edited_cells = selected_cells * editing_eff
    edited_cells_retained = edited_cells * (cells_retained / selected_cells) if selected_cells != 0 else 0.0
    # edited genomes entering PCR estimated by scaling genomes_input by fraction of edited cells in pelleted vs pelleted total
    edited_genomes_PCR = genomes_input * (edited_cells_retained / cells_pelleted) if cells_pelleted != 0 else 0.0

    genomes_per_design = genomes_input / library_size if library_size != 0 else 0.0
    edited_genomes_per_design = genomes_per_design * editing_eff
    reads_per_design = reads_per_replicate / library_size if library_size != 0 else 0.0
    reads_per_edited_genome = reads_per_design / edited_genomes_per_design if edited_genomes_per_design != 0 else np.nan

    # coverage stages per design
    coverage_stages = {
        "Transfected": selected_cells / library_size if library_size != 0 else 0.0,
        "Edited": edited_cells / library_size if library_size != 0 else 0.0,
        "Edited Retained": edited_cells_retained / library_size if library_size != 0 else 0.0,
        "PCR Input": genomes_per_design,
        "Reads": reads_per_design
    }

    return dict(
        selected_cells=selected_cells,
        edited_cells=edited_cells,
        edited_cells_retained=edited_cells_retained,
        edited_genomes_PCR=edited_genomes_PCR,
        genomes_per_design=genomes_per_design,
        edited_genomes_per_design=edited_genomes_per_design,
        reads_per_design=reads_per_design,
        reads_per_edited_genome=reads_per_edited_genome,
        coverage_stages=coverage_stages
    )

def render_main_output(
    DNA_sgRNA_per_million,
    DNA_HDR_per_million,
    cells_transfected,
    transfection_eff,
    editing_eff,
    library_size,
    cells_retained,
    cells_pelleted,
    genomes_input,
    reads_per_replicate
):
    with output_text:
        clear_output(wait=True)
        # totals
        DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
        DNA_HDR_total = DNA_HDR_per_million * (cells_transfected / 1e6)
        mult = 3.1 * 1.1
        DNA_sgRNA_full = DNA_sgRNA_total * mult
        DNA_HDR_full = DNA_HDR_total * mult

        # compute metrics
        m = compute_metrics(cells_transfected, editing_eff, library_size, cells_retained, cells_pelleted, genomes_input, reads_per_replicate)

        # textual report
        print("=== Transfection Inputs ===")
        print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
        print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
        print(f"HDR DNA:   {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)\n")
        print(f"Full screen requirement (3× reps + 0.1 margin + 10% buffer): ×{mult:.2f}")
        print(f"→ sgRNA total required: {DNA_sgRNA_full:.2f} µg")
        print(f"→ HDR total required:   {DNA_HDR_full:.2f} µg\n")

        print("=== Post-Selection Outcomes ===")
        print(f"Editing efficiency: {editing_eff*100:.0f}%")
        print(f"Edited cells retained: {m['edited_cells_retained']/1e6:.2f} M")
        print(f"Edited genomes entering PCR: {m['edited_genomes_PCR']/1e6:.2f} M\n")

        print("=== Coverage Metrics ===")
        print(f"Total genomes per design (PCR input): {m['genomes_per_design']:.1f}")
        print(f"Edited genomes per design: {m['edited_genomes_per_design']:.1f}")
        print(f"Edited retained per design: {m['coverage_stages']['Edited Retained']:.1f}")
        print(f"Total reads per design: {m['reads_per_design']:.1f}")
        rpmg = m['reads_per_edited_genome']
        if np.isnan(rpmg):
            print(f"Reads per edited genome: n/a (no edited genomes present)")
        else:
            print(f"Reads per edited genome: {rpmg:.2f}\n")

    # render coverage plot with annotated values using Matplotlib for consistent style (we also provide Plotly sweep later)
    with coverage_plot_out:
        clear_output(wait=True)
        stages = list(m['coverage_stages'].keys())
        values = list(m['coverage_stages'].values())
        plt.figure(figsize=(7,4.5))
        plt.plot(stages, values, marker='o', lw=2)
        plt.yscale('log')
        plt.ylabel('Coverage per variant (log scale)')
        plt.title('Coverage Across Pipeline Stages')
        plt.grid(True, which="both", ls="--", lw=0.5)

        # annotate points with numeric values (compact formatting)
        for i, (stage, val) in enumerate(zip(stages, values)):
            if val >= 1000:
                label = f"{val:,.0f}"
            elif val >= 1:
                label = f"{val:.1f}"
            else:
                label = f"{val:.2g}"
            plt.text(i, val * 1.05, label, ha='center', va='bottom', fontsize=9)

        plt.show()

# initial render
render_main_output(
    sgRNA_s.value, HDR_s.value,
    cells_transfected_s.value,
    transfection_eff_s.value,
    editing_eff_s.value,
    library_size_s.value,
    cells_retained_s.value,
    cells_pelleted_s.value,
    genomes_input_s.value,
    reads_per_replicate_s.value
)

# connect interactive output so it updates when sliders change
controls_map = {
    'DNA_sgRNA_per_million': sgRNA_s,
    'DNA_HDR_per_million': HDR_s,
    'cells_transfected': cells_transfected_s,
    'transfection_eff': transfection_eff_s,
    'editing_eff': editing_eff_s,
    'library_size': library_size_s,
    'cells_retained': cells_retained_s,
    'cells_pelleted': cells_pelleted_s,
    'genomes_input': genomes_input_s,
    'reads_per_replicate': reads_per_replicate_s
}
out_main = widgets.interactive_output(render_main_output, controls_map)

# ------------------------------
# Scenario Comparator (Tab)
# ------------------------------
# Tab 1: Sweep Plot (Plotly)
param_options = {
    'editing_eff': ('Editing efficiency', editing_eff_s),
    'cells_transfected': ('Cells transfected', cells_transfected_s),
    'genomes_input': ('Genomes into PCR', genomes_input_s),
    'reads_per_replicate': ('Reads per replicate', reads_per_replicate_s),
    'library_size': ('Library size', library_size_s)
}

sweep_param_dd = widgets.Dropdown(options=[(v[0], k) for k,v in param_options.items()],
                                  value='editing_eff', description='Sweep param:')
sweep_points = widgets.IntSlider(value=50, min=10, max=200, step=10, description='Points')
sweep_button = widgets.Button(description='Generate sweep', button_style='primary')

sweep_plot_out = widgets.Output()

def generate_sweep_plot(b):
    with sweep_plot_out:
        clear_output(wait=True)
        key = sweep_param_dd.value
        label, slider_obj = param_options[key][0], param_options[key][1]
        # automatically adapt to slider min/max
        start = slider_obj.min
        stop = slider_obj.max
        # create sweep values (use linspace)
        xvals = np.linspace(start, stop, sweep_points.value)
        edited_per_design = []
        reads_per_edited = []
        reads_per_design_list = []
        for xv in xvals:
            # build scenario using current controls, but replace sweep param
            params = {
                'cells_transfected': cells_transfected_s.value,
                'editing_eff': editing_eff_s.value,
                'library_size': library_size_s.value,
                'cells_retained': cells_retained_s.value,
                'cells_pelleted': cells_pelleted_s.value,
                'genomes_input': genomes_input_s.value,
                'reads_per_replicate': reads_per_replicate_s.value
            }
            params[key] = xv
            m = compute_metrics(params['cells_transfected'], params['editing_eff'], params['library_size'],
                                params['cells_retained'], params['cells_pelleted'], params['genomes_input'],
                                params['reads_per_replicate'])
            edited_per_design.append(m['edited_genomes_per_design'])
            reads_per_edited.append(m['reads_per_edited_genome'])
            reads_per_design_list.append(m['reads_per_design'])

        # Build interactive Plotly figure
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=xvals, y=edited_per_design, mode='lines+markers', name='Edited genomes per design'))
        fig.add_trace(go.Scatter(x=xvals, y=reads_per_edited, mode='lines+markers', name='Reads per edited genome', yaxis='y2'))
        fig.add_trace(go.Scatter(x=xvals, y=reads_per_design_list, mode='lines', name='Reads per design', line=dict(dash='dash')))

        # add vertical line showing current value
        curr = slider_obj.value
        fig.add_vline(x=curr, line=dict(color='gray', dash='dot'), annotation_text='current', annotation_position='top right')

        # axes
        fig.update_layout(
            title=f"Sweep: {label}",
            xaxis_title=label,
            yaxis_title='Edited genomes per design (left)',
            yaxis=dict(type='linear'),
            yaxis2=dict(title='Reads per edited genome (right)', overlaying='y', side='right'),
            legend=dict(x=0.01, y=0.99)
        )
        fig.update_xaxes(type='linear')
        fig.update_traces(marker=dict(size=6))
        fig.show()

sweep_button.on_click(generate_sweep_plot)

sweep_controls = widgets.VBox([sweep_param_dd, sweep_points, sweep_button, sweep_plot_out])

# Tab 2: Scenario Table (up to 3 scenarios)
# We'll allow user to edit: cells_transfected, editing_eff, genomes_input, reads_per_replicate, library_size
def scenario_row_defaults():
    return {
        'cells_transfected': cells_transfected_s.value,
        'editing_eff': editing_eff_s.value,
        'library_size': library_size_s.value,
        'genomes_input': genomes_input_s.value,
        'reads_per_replicate': reads_per_replicate_s.value
    }

def make_scenario_widget(prefix, defaults):
    w_cells = widgets.FloatText(value=defaults['cells_transfected'], description='Cells', layout=widgets.Layout(width='240px'))
    w_edit = widgets.FloatText(value=defaults['editing_eff'], description='Edit eff', layout=widgets.Layout(width='240px'))
    w_lib  = widgets.IntText(value=int(defaults['library_size']), description='Library', layout=widgets.Layout(width='240px'))
    w_genomes = widgets.FloatText(value=defaults['genomes_input'], description='Genomes in PCR', layout=widgets.Layout(width='240px'))
    w_reads = widgets.FloatText(value=defaults['reads_per_replicate'], description='Reads', layout=widgets.Layout(width='240px'))
    box = widgets.VBox([widgets.HTML(f"<b>{prefix}</b>"), w_cells, w_edit, w_lib, w_genomes, w_reads])
    return {
        'box': box,
        'cells': w_cells,
        'edit': w_edit,
        'lib': w_lib,
        'genomes': w_genomes,
        'reads': w_reads
    }

scenario1 = make_scenario_widget('Scenario A', scenario_row_defaults())
scenario2 = make_scenario_widget('Scenario B', scenario_row_defaults())
scenario3 = make_scenario_widget('Scenario C', scenario_row_defaults())

compare_button = widgets.Button(description='Compare scenarios', button_style='success')
scenario_out = widgets.Output()

def compare_scenarios(b):
    with scenario_out:
        clear_output(wait=True)
        scenarios = [scenario1, scenario2, scenario3]
        rows = []
        cols = []
        for i, s in enumerate(scenarios):
            params = {
                'cells_transfected': float(s['cells'].value),
                'editing_eff': float(s['edit'].value),
                'library_size': float(s['lib'].value),
                'cells_retained': float(cells_retained_s.value),
                'cells_pelleted': float(cells_pelleted_s.value),
                'genomes_input': float(s['genomes'].value),
                'reads_per_replicate': float(s['reads'].value)
            }
            m = compute_metrics(params['cells_transfected'], params['editing_eff'], params['library_size'],
                                params['cells_retained'], params['cells_pelleted'], params['genomes_input'],
                                params['reads_per_replicate'])
            col = {
                'Edited cells retained (M)': m['edited_cells_retained']/1e6,
                'Edited genomes entering PCR (M)': m['edited_genomes_PCR']/1e6,
                'Genomes per design': m['genomes_per_design'],
                'Edited genomes per design': m['edited_genomes_per_design'],
                'Reads per design': m['reads_per_design'],
                'Reads per edited genome': m['reads_per_edited_genome']
            }
            cols.append(col)
            cols[-1]['Scenario'] = f"Scenario {chr(ord('A')+i)}"
        # DataFrame
        df = pd.DataFrame(cols).set_index('Scenario').T
        display(df.style.format("{:.3g}"))
        # Bar chart: Reads per edited genome comparison
        fig = go.Figure()
        for scenario_idx, colname in enumerate(['Scenario A','Scenario B','Scenario C']):
            if colname in df.columns:
                fig.add_trace(go.Bar(name=colname, x=df.index, y=df[colname]))
        fig.update_layout(barmode='group', title='Scenario metrics (per-index values)')
        fig.show()

compare_button.on_click(compare_scenarios)

scenarios_box = widgets.HBox([scenario1['box'], scenario2['box'], scenario3['box']])
scenario_tab_content = widgets.VBox([scenarios_box, compare_button, scenario_out])

# Build Tab widget
tab = widgets.Tab(children=[sweep_controls, scenario_tab_content])
tab.set_title(0, 'Sweep Plot')
tab.set_title(1, 'Scenario Table')

# ------------------------------
# Final layout: main panel + comparator
# ------------------------------
top = widgets.VBox([
    widgets.HTML("<h2>SGE Screen Interactive Model</h2>"),
    widgets.HTML("<h3>DNA Inputs</h3>"),
    sgRNA_box,
    HDR_box,
    ratio_box,
    widgets.HBox([sgRNA_full_label, HDR_full_label]),
    widgets.HTML("<h3>Cell & Editing Parameters</h3>"),
    cells_transfected_box,
    transfection_eff_box,
    editing_eff_box,
    library_size_box,
    cells_retained_box,
    cells_pelleted_box,
    genomes_input_box,
    reads_per_replicate_box
])

display(top)
display(out_main)           # interactive output connected to sliders
display(output_text)        # textual report
display(coverage_plot_out)  # matplotlib coverage plot
display(tab)                # comparator (plotly + scenario table)


Output()

Output()

Output()

# Section 8


---


*   Sweep plot export not working






In [ ]:
# @title
# ===========================
#    SGE SCREEN MODEL — INTERACTIVE (Plotly)
# ===========================
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------
# Helper: slider + textbox pair
# ------------------------------
def linked_slider_text(value, minv, maxv, step, description, fmt='.3f', width='560px'):
    slider = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description,
        style={'description_width': '300px'},
        layout=widgets.Layout(width=width),
        readout_format=fmt
    )
    text = widgets.FloatText(value=value, layout=widgets.Layout(width='120px'))
    widgets.jslink((slider, 'value'), (text, 'value'))
    return slider, text, widgets.HBox([slider, text])

# ------------------------------
# Core sliders (tune boundaries here)
# ------------------------------
cells_transfected_s, cells_transfected_t, cells_transfected_box = linked_slider_text(20e6, 11e6, 50e6, 1e6, 'Cells transfected', fmt='.2e')
transfection_eff_s, transfection_eff_t, transfection_eff_box = linked_slider_text(0.6, 0.05, 0.9, 0.05, 'Transfection eff. (ref)', fmt='.2f')
editing_eff_s, editing_eff_t, editing_eff_box = linked_slider_text(0.4, 0.05, 0.9, 0.05, 'Editing efficiency', fmt='.2f')
library_size_s, library_size_t, library_size_box = linked_slider_text(2500, 500, 5000, 100, 'Library size', fmt='.0f')
cells_retained_s, cells_retained_t, cells_retained_box = linked_slider_text(5e6, 1e6, 10e6, 0.5e6, 'Cells retained', fmt='.2e')
cells_pelleted_s, cells_pelleted_t, cells_pelleted_box = linked_slider_text(3e6, 1e6, 10e6, 0.5e6, 'Cells pelleted', fmt='.2e')
genomes_input_s, genomes_input_t, genomes_input_box = linked_slider_text(1.52e6, 0.5e6, 5e6, 0.1e6, 'Genomes into PCR', fmt='.2e')
reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box = linked_slider_text(10e6, 1e6, 20e6, 1e6, 'Reads per replicate', fmt='.2e')

# ------------------------------
# DNA sliders (per-million ↔ total linked)
# ------------------------------
def make_dna_pair(label, default_per_million, minp, maxp, step):
    perM_slider, perM_text, perM_box = linked_slider_text(default_per_million, minp, maxp, step, f'{label} DNA (µg / 10⁶ cells)')
    total_text = widgets.FloatText(
        value=default_per_million * (cells_transfected_s.value / 1e6),
        description=f'{label} total (µg)',
        style={'description_width': '180px'},
        layout=widgets.Layout(width='320px')
    )

    def update_total(*args):
        total_text.value = perM_slider.value * (cells_transfected_s.value / 1e6)
    perM_slider.observe(update_total, 'value')
    cells_transfected_s.observe(update_total, 'value')

    def update_perM(change):
        # avoid division by zero
        denom = (cells_transfected_s.value / 1e6) if cells_transfected_s.value != 0 else 1e-6
        perM_slider.value = total_text.value / denom
    total_text.observe(update_perM, 'value')

    return perM_slider, perM_text, total_text, widgets.HBox([perM_slider, perM_text, total_text])

sgRNA_s, sgRNA_t, sgRNA_total_t, sgRNA_box = make_dna_pair('sgRNA', 0.375, 0.1, 1.0, 0.05)
HDR_s, HDR_t, HDR_total_t, HDR_box = make_dna_pair('HDR', 0.75, 0.1, 1.5, 0.05)

# ------------------------------
# HDR : sgRNA ratio linking
# ------------------------------
ratio_slider, ratio_text, ratio_box = linked_slider_text(
    value=HDR_s.value / sgRNA_s.value,
    minv=0.5, maxv=4.0, step=0.05,
    description='HDR : sgRNA ratio',
    fmt='.2f'
)

_updating = {'ratio': False, 'sgRNA': False, 'HDR': False}

def update_ratio(*args):
    if _updating['ratio']: return
    _updating['ratio'] = True
    # guard division by zero
    if sgRNA_s.value == 0:
        ratio = np.nan
    else:
        ratio = HDR_s.value / sgRNA_s.value
    ratio_slider.value = ratio
    ratio_text.value = ratio
    _updating['ratio'] = False

def update_HDR_from_ratio(*args):
    if _updating['HDR']: return
    _updating['HDR'] = True
    HDR_s.value = sgRNA_s.value * ratio_slider.value
    _updating['HDR'] = False
    update_ratio()

def update_from_HDR(*args):
    if _updating['sgRNA']: return
    _updating['sgRNA'] = True
    # update ratio based on new HDR value
    if sgRNA_s.value == 0:
        r = np.nan
    else:
        r = HDR_s.value / sgRNA_s.value
    ratio_slider.value = r
    ratio_text.value = r
    _updating['sgRNA'] = False
    update_ratio()

sgRNA_s.observe(update_HDR_from_ratio, 'value')
HDR_s.observe(update_from_HDR, 'value')
ratio_slider.observe(update_HDR_from_ratio, 'value')
ratio_text.observe(update_HDR_from_ratio, 'value')

# ------------------------------
# Live labels for full-screen DNA requirements
# ------------------------------
sgRNA_full_label = widgets.HTML()
HDR_full_label = widgets.HTML()

def update_full_screen_labels():
    per_run_sg = sgRNA_s.value * (cells_transfected_s.value / 1e6)
    per_run_hdr = HDR_s.value * (cells_transfected_s.value / 1e6)
    mult = 3.1 * 1.1
    sg_full = per_run_sg * mult
    hdr_full = per_run_hdr * mult
    sgRNA_full_label.value = f"<b>sgRNA per run:</b> {per_run_sg:.2f} µg &nbsp;&nbsp; <b>full screen (×{mult:.2f}):</b> {sg_full:.2f} µg"
    HDR_full_label.value = f"<b>HDR per run:</b> {per_run_hdr:.2f} µg &nbsp;&nbsp; <b>full screen (×{mult:.2f}):</b> {hdr_full:.2f} µg"

# attach updater observers
sgRNA_s.observe(lambda *_: update_full_screen_labels(), 'value')
HDR_s.observe(lambda *_: update_full_screen_labels(), 'value')
cells_transfected_s.observe(lambda *_: update_full_screen_labels(), 'value')
update_full_screen_labels()

# ------------------------------
# Output areas
# ------------------------------
output_text = widgets.Output()
coverage_plot_out = widgets.Output()

# ------------------------------
# Core model computation & display (text + matplotlib coverage)
# ------------------------------
def compute_metrics(cells_transfected, editing_eff, library_size, cells_retained, cells_pelleted, genomes_input, reads_per_replicate):
    # quantities (absolute)
    selected_cells = cells_transfected
    edited_cells = selected_cells * editing_eff
    edited_cells_retained = edited_cells * (cells_retained / selected_cells) if selected_cells != 0 else 0.0
    # edited genomes entering PCR estimated by scaling genomes_input by fraction of edited cells in pelleted vs pelleted total
    edited_genomes_PCR = genomes_input * (edited_cells_retained / cells_pelleted) if cells_pelleted != 0 else 0.0

    genomes_per_design = genomes_input / library_size if library_size != 0 else 0.0
    edited_genomes_per_design = genomes_per_design * editing_eff
    reads_per_design = reads_per_replicate / library_size if library_size != 0 else 0.0
    reads_per_edited_genome = reads_per_design / edited_genomes_per_design if edited_genomes_per_design != 0 else np.nan

    # coverage stages per design
    coverage_stages = {
        "Transfected": selected_cells / library_size if library_size != 0 else 0.0,
        "Edited": edited_cells / library_size if library_size != 0 else 0.0,
        "Edited Retained": edited_cells_retained / library_size if library_size != 0 else 0.0,
        "PCR Input": genomes_per_design,
        "Reads": reads_per_design
    }

    return dict(
        selected_cells=selected_cells,
        edited_cells=edited_cells,
        edited_cells_retained=edited_cells_retained,
        edited_genomes_PCR=edited_genomes_PCR,
        genomes_per_design=genomes_per_design,
        edited_genomes_per_design=edited_genomes_per_design,
        reads_per_design=reads_per_design,
        reads_per_edited_genome=reads_per_edited_genome,
        coverage_stages=coverage_stages
    )

def render_main_output(
    DNA_sgRNA_per_million,
    DNA_HDR_per_million,
    cells_transfected,
    transfection_eff,
    editing_eff,
    library_size,
    cells_retained,
    cells_pelleted,
    genomes_input,
    reads_per_replicate
):
    with output_text:
        clear_output(wait=True)
        # totals
        DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
        DNA_HDR_total = DNA_HDR_per_million * (cells_transfected / 1e6)
        mult = 3.1 * 1.1
        DNA_sgRNA_full = DNA_sgRNA_total * mult
        DNA_HDR_full = DNA_HDR_total * mult

        # compute metrics
        m = compute_metrics(cells_transfected, editing_eff, library_size, cells_retained, cells_pelleted, genomes_input, reads_per_replicate)

        # textual report
        print("=== Transfection Inputs ===")
        print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
        print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
        print(f"HDR DNA:   {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)\n")
        print(f"Full screen requirement (3× reps + 0.1 margin + 10% buffer): ×{mult:.2f}")
        print(f"→ sgRNA total required: {DNA_sgRNA_full:.2f} µg")
        print(f"→ HDR total required:   {DNA_HDR_full:.2f} µg\n")

        print("=== Post-Selection Outcomes ===")
        print(f"Editing efficiency: {editing_eff*100:.0f}%")
        print(f"Edited cells retained: {m['edited_cells_retained']/1e6:.2f} M")
        print(f"Edited genomes entering PCR: {m['edited_genomes_PCR']/1e6:.2f} M\n")

        print("=== Coverage Metrics ===")
        print(f"Total genomes per design (PCR input): {m['genomes_per_design']:.1f}")
        print(f"Edited genomes per design: {m['edited_genomes_per_design']:.1f}")
        print(f"Edited retained per design: {m['coverage_stages']['Edited Retained']:.1f}")
        print(f"Total reads per design: {m['reads_per_design']:.1f}")
        rpmg = m['reads_per_edited_genome']
        if np.isnan(rpmg):
            print(f"Reads per edited genome: n/a (no edited genomes present)")
        else:
            print(f"Reads per edited genome: {rpmg:.2f}\n")

    # render coverage plot with annotated values using Matplotlib for consistent style (we also provide Plotly sweep later)
    with coverage_plot_out:
        clear_output(wait=True)
        stages = list(m['coverage_stages'].keys())
        values = list(m['coverage_stages'].values())
        plt.figure(figsize=(7,4.5))
        plt.plot(stages, values, marker='o', lw=2)
        plt.yscale('log')
        plt.ylabel('Coverage per variant (log scale)')
        plt.title('Coverage Across Pipeline Stages')
        plt.grid(True, which="both", ls="--", lw=0.5)

        # annotate points with numeric values (compact formatting)
        for i, (stage, val) in enumerate(zip(stages, values)):
            if val >= 1000:
                label = f"{val:,.0f}"
            elif val >= 1:
                label = f"{val:.1f}"
            else:
                label = f"{val:.2g}"
            plt.text(i, val * 1.05, label, ha='center', va='bottom', fontsize=9)

        plt.show()

# initial render
render_main_output(
    sgRNA_s.value, HDR_s.value,
    cells_transfected_s.value,
    transfection_eff_s.value,
    editing_eff_s.value,
    library_size_s.value,
    cells_retained_s.value,
    cells_pelleted_s.value,
    genomes_input_s.value,
    reads_per_replicate_s.value
)

# connect interactive output so it updates when sliders change
controls_map = {
    'DNA_sgRNA_per_million': sgRNA_s,
    'DNA_HDR_per_million': HDR_s,
    'cells_transfected': cells_transfected_s,
    'transfection_eff': transfection_eff_s,
    'editing_eff': editing_eff_s,
    'library_size': library_size_s,
    'cells_retained': cells_retained_s,
    'cells_pelleted': cells_pelleted_s,
    'genomes_input': genomes_input_s,
    'reads_per_replicate': reads_per_replicate_s
}
out_main = widgets.interactive_output(render_main_output, controls_map)

# ------------------------------
# Scenario Comparator (Tab)
# ============================
# 📊 Sweep Plot (Plotly Colab HTML Fallback — Fully Working)
# ============================
# import numpy as np # Already imported
# import plotly.graph_objects as go # Already imported
# import ipywidgets as widgets # Already imported
from IPython.display import HTML # Already imported

# Define the output widget for the sweep plot
output_sweep = widgets.Output()

# --- Dropdown and Button ---
sweep_param_dd = widgets.Dropdown(
    options=[
        ('Editing efficiency', 'editing_eff'),
        ('Cells transfected', 'cells_transfected'),
        ('Genomes input (PCR)', 'genomes_input'),
        ('Reads per replicate', 'reads_per_replicate'),
        ('Library size', 'library_size') # Added Library size to options
    ],
    value='editing_eff',
    description='Sweep parameter:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

sweep_button = widgets.Button(
    description='Generate Sweep',
    button_style='info',
    tooltip='Generate sweep plot across parameter range'
)

# --- Callback for sweep generation ---
def run_sweep(b):
    with output_sweep: # Direct all output to this widget
        clear_output(wait=True) # Clear only THIS output widget

        param = sweep_param_dd.value

        slider_map = {
            'editing_eff': editing_eff_s,
            'cells_transfected': cells_transfected_s,
            'genomes_input': genomes_input_s,
            'reads_per_replicate': reads_per_replicate_s,
            'library_size': library_size_s # Added to map
        }
        slider = slider_map[param]

        # Use linspace for smoother sweep, number of points can be made adjustable
        sweep_range = np.linspace(slider.min, slider.max, 50)
        edited_genomes, reads_per_edited, reads_per_design = [], [], []

        for val in sweep_range:
            kwargs = dict(
                # Use the global slider values for fixed parameters
                DNA_sgRNA_per_million=sgRNA_s.value,
                DNA_HDR_per_million=HDR_s.value,
                cells_transfected=cells_transfected_s.value,
                transfection_eff=transfection_eff_s.value,
                editing_eff=editing_eff_s.value,
                library_size=library_size_s.value,
                cells_retained=cells_retained_s.value,
                cells_pelleted=cells_pelleted_s.value,
                genomes_input=genomes_input_s.value,
                reads_per_replicate=reads_per_replicate_s.value
            )
            kwargs[param] = val

            # Recalculate metrics using compute_metrics
            m = compute_metrics(kwargs['cells_transfected'], kwargs['editing_eff'], kwargs['library_size'],
                                kwargs['cells_retained'], kwargs['cells_pelleted'], kwargs['genomes_input'],
                                kwargs['reads_per_replicate'])

            edited_genomes.append(m['edited_genomes_per_design'])
            reads_per_edited.append(m['reads_per_edited_genome'])
            reads_per_design.append(m['reads_per_design'])

        # --- Build Plotly figure ---
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=sweep_range, y=edited_genomes,
            mode='lines+markers',
            name='Edited genomes per design',
            line=dict(color='royalblue')
        ))

        fig.add_trace(go.Scatter(
            x=sweep_range, y=reads_per_edited,
            mode='lines+markers',
            name='Reads per edited genome',
            line=dict(color='darkorange')
        ))

        fig.add_trace(go.Scatter(
            x=sweep_range, y=reads_per_design,
            mode='lines+markers', # Changed to lines+markers for consistency
            name='Reads per design (constant)',
            line=dict(color='gray', dash='dot')
        ))

        # Add vertical line for current value
        current_val = slider.value
        fig.add_vline(
            x=current_val,
            line_dash="dash",
            line_color="red",
            annotation_text=f"Current = {current_val:.2g}",
            annotation_position="top right"
        )

        # Update layout with dynamic titles
        # Get the full description from the dropdown options list
        param_description = sweep_param_dd.options[sweep_param_dd.index][0]
        fig.update_layout(
            title=f"Sweep of {param_description}",
            xaxis_title=param_description,
            yaxis_title="Coverage Metric Value",
            template="plotly_white",
            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
        )

        # ✅ Render figure explicitly as inline HTML — always works in Colab
        html = fig.to_html(include_plotlyjs='cdn', full_html=False)
        display(HTML(html))

# --- Hook up button ---
sweep_button.on_click(run_sweep)

# --- Define sweep_tab: This is the missing piece for the NameError ---
sweep_tab = widgets.VBox([
    widgets.HBox([sweep_param_dd, sweep_button]),
    output_sweep # The output widget defined above where the plot will be rendered
])

# (Removed the standalone display(widgets.HBox([sweep_param_dd, sweep_button])) from the original code)

# -----------------------------
# Scenario Table Tab
# -----------------------------
output_table = widgets.Output()

def scenario_row_defaults():
    return {
        'cells_transfected': cells_transfected_s.value,
        'editing_eff': editing_eff_s.value,
        'library_size': library_size_s.value,
        'genomes_input': genomes_input_s.value,
        'reads_per_replicate': reads_per_replicate_s.value
    }

def make_scenario_widget(prefix, defaults):
    w_cells = widgets.FloatText(value=defaults['cells_transfected'], description='Cells', layout=widgets.Layout(width='240px'))
    w_edit = widgets.FloatText(value=defaults['editing_eff'], description='Edit eff', layout=widgets.Layout(width='240px'))
    w_lib  = widgets.IntText(value=int(defaults['library_size']), description='Library', layout=widgets.Layout(width='240px'))
    w_genomes = widgets.FloatText(value=defaults['genomes_input'], description='Genomes in PCR', layout=widgets.Layout(width='240px'))
    w_reads = widgets.FloatText(value=defaults['reads_per_replicate'], description='Reads', layout=widgets.Layout(width='240px'))
    box = widgets.VBox([widgets.HTML(f"<b>{prefix}</b>"), w_cells, w_edit, w_lib, w_genomes, w_reads])
    return {
        'box': box,
        'cells': w_cells,
        'edit': w_edit,
        'lib': w_lib,
        'genomes': w_genomes,
        'reads': w_reads
    }

scenario1 = make_scenario_widget('Scenario A', scenario_row_defaults())
scenario2 = make_scenario_widget('Scenario B', scenario_row_defaults())
scenario3 = make_scenario_widget('Scenario C', scenario_row_defaults())

compare_button = widgets.Button(description='Compare scenarios', button_style='success')

def compare_scenarios(b):
    with output_table:
        clear_output(wait=True)
        scenarios = [scenario1, scenario2, scenario3]
        cols = []
        for i, s in enumerate(scenarios):
            params = {
                'cells_transfected': float(s['cells'].value),
                'editing_eff': float(s['edit'].value),
                'library_size': float(s['lib'].value),
                'cells_retained': float(cells_retained_s.value),
                'cells_pelleted': float(cells_pelleted_s.value),
                'genomes_input': float(s['genomes'].value),
                'reads_per_replicate': float(s['reads'].value)
            }
            m = compute_metrics(params['cells_transfected'], params['editing_eff'], params['library_size'],
                                params['cells_retained'], params['cells_pelleted'], params['genomes_input'],
                                params['reads_per_replicate'])
            col = {
                'Edited cells retained (M)': m['edited_cells_retained']/1e6,
                'Edited genomes entering PCR (M)': m['edited_genomes_PCR']/1e6,
                'Genomes per design': m['genomes_per_design'],
                'Edited genomes per design': m['edited_genomes_per_design'],
                'Reads per design': m['reads_per_design'],
                'Reads per edited genome': m['reads_per_edited_genome']
            }
            cols.append(col)
            cols[-1]['Scenario'] = f"Scenario {chr(ord('A')+i)}"
        # DataFrame
        df = pd.DataFrame(cols).set_index('Scenario').T
        display(df.style.format("{:.3g}"))

        # Bar chart: Reads per edited genome comparison
        fig = go.Figure()
        metrics_to_plot = ['Edited genomes per design', 'Reads per design', 'Reads per edited genome']
        for metric in metrics_to_plot:
             for scenario_idx, colname in enumerate(['Scenario A','Scenario B','Scenario C']):
                if colname in df.columns:
                     fig.add_trace(go.Bar(name=f'{colname} - {metric}', x=[metric], y=[df.loc[metric, colname]]))

        fig.update_layout(barmode='group', title='Scenario metrics (per-design values)')
        fig.show()


compare_button.on_click(compare_scenarios)

scenarios_box = widgets.HBox([scenario1['box'], scenario2['box'], scenario3['box']])
scenario_tab_content = widgets.VBox([scenarios_box, compare_button, output_table])

# -----------------------------
# Tabs: Sweep + Table
# -----------------------------
tabs = widgets.Tab(children=[sweep_tab, scenario_tab_content])
tabs.set_title(0, 'Sweep Plot')
tabs.set_title(1, 'Scenario Table')

# -----------------------------
# Display Layout
# -----------------------------
ui = widgets.VBox([
    widgets.HTML("<h2>SGE Screen Interactive Model</h2>"),
    widgets.HTML("<h3>DNA Inputs</h3>"),
    widgets.HBox([sgRNA_s, sgRNA_t, sgRNA_box]), # Use sgRNA_s, sgRNA_t, sgRNA_box from make_dna_pair
    widgets.HBox([HDR_s, HDR_t, HDR_box, ratio_box]), # Use HDR_s, HDR_t, HDR_box, ratio_box from make_dna_pair and linked_slider_text
    widgets.HTML("<h3>Cell & Editing Parameters</h3>"),
    widgets.HBox([cells_transfected_s, cells_transfected_t, cells_transfected_box]),
    widgets.HBox([transfection_eff_s, transfection_eff_t, transfection_eff_box, # Use existing _s, _t, _box variables
                  editing_eff_s, editing_eff_t, editing_eff_box]),
    widgets.HBox([library_size_s, library_size_t, library_size_box]),
    widgets.HBox([cells_retained_s, cells_retained_t, cells_retained_box,
                  cells_pelleted_s, cells_pelleted_t, cells_pelleted_box]),
    widgets.HBox([genomes_input_s, genomes_input_t, genomes_input_box,
                  reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box]),
    output_text, # Use existing output widgets
    coverage_plot_out, # Use existing output widgets
    tabs
])
display(ui)


# Section 9



*   Sweep plot export now working
*   Interactive input disappeared?





In [ ]:
# @title
# ===============================
# 📈 Sweep Export (Plotly HTML Fallback)
# ===============================
import numpy as np
import plotly.graph_objects as go
from google.colab import files
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# --- Sweep parameter controls ---
sweep_param_dd = widgets.Dropdown(
    options=[
        ('Editing efficiency', 'editing_eff'),
        ('Cells transfected', 'cells_transfected'),
        ('Genomes input (PCR)', 'genomes_input'),
        ('Reads per replicate', 'reads_per_replicate'),
        ('Library size', 'library_size') # Added Library size
    ],
    value='editing_eff',
    description='Sweep parameter:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': 'initial'}
)

sweep_points = widgets.IntSlider(value=50, min=10, max=500, step=10, description='Points')
yscale_dd = widgets.Dropdown(options=['linear', 'log'], value='linear', description='Y-scale')
export_button = widgets.Button(description='Generate & Export Sweep', button_style='success')

output_export = widgets.Output(layout=widgets.Layout(border='1px solid #ccc'))

def run_sweep_export(b):
    with output_export:
        clear_output(wait=True)
        param = sweep_param_dd.value

        # Map parameter name → associated slider (must exist in your model)
        slider_map = {
            'editing_eff': editing_eff_s,
            'cells_transfected': cells_transfected_s,
            'genomes_input': genomes_input_s,
            'reads_per_replicate': reads_per_replicate_s,
            'library_size': library_size_s # Added Library size to map
        }
        slider = slider_map[param]

        # Sweep range using slider limits
        x = np.linspace(slider.min, slider.max, sweep_points.value)

        edited_genomes = []
        reads_per_edited = []
        reads_per_design = []

        # Get current values of fixed parameters
        fixed_params = {
            'DNA_sgRNA_per_million': DNA_sgRNA_per_million_s.value,
            'DNA_HDR_per_million': DNA_HDR_per_million_s.value,
            'cells_transfected': cells_transfected_s.value,
            'transfection_eff': transfection_eff_s.value,
            'editing_eff': editing_eff_s.value,
            'library_size': library_size_s.value,
            'cells_retained': cells_retained_s.value,
            'cells_pelleted': cells_pelleted_s.value,
            'genomes_input': genomes_input_s.value,
            'reads_per_replicate': reads_per_replicate_s.value
        }


        for val in x:
            kwargs = fixed_params.copy() # Start with fixed params
            kwargs[param] = val # Vary the sweep parameter

            genomes_per_variant = kwargs['genomes_input'] / kwargs['library_size']
            edited_per_design = genomes_per_variant * kwargs['editing_eff']
            reads_per_variant = kwargs['reads_per_replicate'] / kwargs['library_size']
            reads_per_edited_val = reads_per_variant / (edited_per_design) if edited_per_design !=0 else np.nan

            edited_genomes.append(edited_per_design)
            reads_per_edited.append(reads_per_edited_val)
            reads_per_design.append(reads_per_variant)

        # --- Build Plotly figure ---
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x, y=edited_genomes, mode='lines+markers',
            name='Edited genomes per design'
        ))
        fig.add_trace(go.Scatter(
            x=x, y=reads_per_design, mode='lines+markers',
            name='Reads per design'
        ))
        fig.add_trace(go.Scatter(
            x=x, y=reads_per_edited, mode='lines+markers',
            name='Reads per edited genome',
            yaxis='y2'
        ))

        fig.update_layout(
            title=f"Sweep: {sweep_param_dd.label if hasattr(sweep_param_dd, 'label') else sweep_param_dd.description}",
            xaxis_title=param,
            yaxis=dict(title='Genomes / Reads per design', type=yscale_dd.value),
            yaxis2=dict(title='Reads per edited genome', overlaying='y', side='right', type=yscale_dd.value),
            template='plotly_white',
            legend=dict(x=0.01, y=0.99),
            width=900, # Revert width and margin as annotation is not in the plot area
            height=500,
            margin=dict(r=50)
        )

        # --- Display fixed parameters and assumptions below the plot ---
        print("<b>Fixed Parameters:</b>")
        for key, value in fixed_params.items():
            if key != param: # Don't show the sweep parameter as fixed
                print(f"- {key}: {value:.2g}")
        print("<br><b>Assumptions:</b>")
        print("- All surviving cells are transfected after selection.")
        print("- Edited genomes entering PCR scale with edited cells in pelleted sample.")
        print("- Full screen requirement includes 3 replicates + 0.1 margin + 10% buffer.")

        # --- Save & offer download ---
        fn = '/content/sweep_plot.html'
        fig.write_html(fn, include_plotlyjs='cdn')
        print("✅ Sweep plot generated and saved as 'sweep_plot.html'.")
        files.download(fn)  # triggers download popup in Colab

        # Display the plot within the output widget
        display(fig)


# Connect button
export_button.on_click(run_sweep_export)

# Display the sweep controls and the output widget
display(widgets.VBox([
    widgets.HTML("<b>📊 Sweep Export Tool (Plotly HTML)</b>"),
    widgets.HBox([sweep_param_dd, sweep_points, yscale_dd, export_button]),
    output_export # Display the output widget
]))

#Section 10



*   Contains both interactive inputs and correct sweep plot generation



In [ ]:
# @title
# =============================================
#     SGE SCREEN TOY MODEL — FULL INTERACTIVE UI (Consolidated)
# =============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
import plotly.graph_objects as go
import plotly.express as px
from google.colab import files

# ------------------------------------------------------
# Helper: slider + textbox pair (two-way linked)
# ------------------------------------------------------
def linked_slider_text(value, minv, maxv, step, description, fmt='.3f', width='560px'):
    slider = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description,
        style={'description_width': '300px'},
        layout=widgets.Layout(width=width),
        readout_format=fmt
    )
    text = widgets.FloatText(value=value, layout=widgets.Layout(width='120px'))
    widgets.jslink((slider, 'value'), (text, 'value'))
    return slider, text, widgets.HBox([slider, text])

# ------------------------------------------------------
# Core sliders (tune boundaries here)
# ------------------------------------------------------
cells_transfected_s, cells_transfected_t, cells_transfected_box = linked_slider_text(20e6, 11e6, 50e6, 1e6, 'Cells transfected', fmt='.2e')
transfection_eff_s, transfection_eff_t, transfection_eff_box = linked_slider_text(0.6, 0.05, 0.9, 0.05, 'Transfection eff. (ref)', fmt='.2f')
editing_eff_s, editing_eff_t, editing_eff_box = linked_slider_text(0.4, 0.05, 0.9, 0.05, 'Editing efficiency', fmt='.2f')
library_size_s, library_size_t, library_size_box = linked_slider_text(2500, 500, 5000, 100, 'Library size', fmt='.0f')
cells_retained_s, cells_retained_t, cells_retained_box = linked_slider_text(5e6, 1e6, 10e6, 0.5e6, 'Cells retained', fmt='.2e')
cells_pelleted_s, cells_pelleted_t, cells_pelleted_box = linked_slider_text(3e6, 1e6, 10e6, 0.5e6, 'Cells pelleted', fmt='.2e')
genomes_input_s, genomes_input_t, genomes_input_box = linked_slider_text(1.52e6, 0.5e6, 5e6, 0.1e6, 'Genomes into PCR', fmt='.2e')
reads_per_replicate_s, reads_per_replicate_t, reads_per_replicate_box = linked_slider_text(10e6, 1e6, 20e6, 1e6, 'Reads per replicate', fmt='.2e')

# ------------------------------------------------------
# DNA sliders (per-million ↔ total linked)
# ------------------------------------------------------
def make_dna_pair(label, default_per_million, minp, maxp, step):
    perM_slider, perM_text, _ = linked_slider_text(
        default_per_million, minp, maxp, step, f'{label} DNA (µg / 10⁶ cells)'
    )
    total_text = widgets.FloatText(
        value=default_per_million * (cells_transfected_s.value / 1e6),
        description=f'{label} total (µg)',
        style={'description_width': '180px'},
        layout=widgets.Layout(width='320px')
    )

    def update_total(*args):
        total_text.value = perM_slider.value * (cells_transfected_s.value / 1e6)
    perM_slider.observe(update_total, 'value')
    cells_transfected_s.observe(update_total, 'value')

    def update_perM(change):
        # avoid division by zero
        denom = (cells_transfected_s.value / 1e6) if cells_transfected_s.value != 0 else 1e-6
        perM_slider.value = total_text.value / denom
    total_text.observe(update_perM, 'value')

    return perM_slider, perM_text, total_text, widgets.HBox([perM_slider, perM_text, total_text])

sgRNA_s, sgRNA_t, sgRNA_total_t, sgRNA_box = make_dna_pair('sgRNA', 0.375, 0.1, 1.0, 0.05)
HDR_s, HDR_t, HDR_total_t, HDR_box = make_dna_pair('HDR', 0.75, 0.1, 1.5, 0.05)

# ------------------------------------------------------
# HDR : sgRNA ratio linking
# ------------------------------------------------------
ratio_slider, ratio_text, ratio_box = linked_slider_text(
    value=HDR_s.value / sgRNA_s.value if sgRNA_s.value != 0 else 0, # Initial check for division by zero
    minv=0.5, maxv=4.0, step=0.05,
    description='HDR : sgRNA ratio',
    fmt='.2f'
)

_updating = {'ratio': False, 'sgRNA': False, 'HDR': False}

def update_ratio(*args):
    if _updating['ratio']: return
    _updating['ratio'] = True
    # guard division by zero
    if sgRNA_s.value == 0:
        ratio = np.nan
    else:
        ratio = HDR_s.value / sgRNA_s.value
    ratio_slider.value = ratio
    ratio_text.value = ratio
    _updating['ratio'] = False

def update_HDR_from_ratio(*args):
    if _updating['HDR']: return
    _updating['HDR'] = True
    HDR_s.value = sgRNA_s.value * ratio_slider.value
    _updating['HDR'] = False
    update_ratio() # Update the ratio display after HDR changes

def update_from_HDR(*args):
    if _updating['sgRNA']: return
    _updating['sgRNA'] = True
    # update ratio based on new HDR value
    if sgRNA_s.value == 0:
        r = np.nan
    else:
        r = HDR_s.value / sgRNA_s.value
    ratio_slider.value = r
    ratio_text.value = r
    _updating['sgRNA'] = False
    update_ratio() # Update the ratio display after HDR changes

sgRNA_s.observe(update_HDR_from_ratio, 'value')
HDR_s.observe(update_from_HDR, 'value')
ratio_slider.observe(update_HDR_from_ratio, 'value')
ratio_text.observe(update_HDR_from_ratio, 'value')

# ------------------------------------------------------
# Live labels for full-screen DNA requirements
# ------------------------------------------------------
sgRNA_full_label = widgets.HTML()
HDR_full_label = widgets.HTML()

def update_full_screen_labels(*args): # Added *args to handle observer arguments
    per_run_sg = sgRNA_s.value * (cells_transfected_s.value / 1e6)
    per_run_hdr = HDR_s.value * (cells_transfected_s.value / 1e6)
    mult = 3.1 * 1.1 # 3 replicates + 0.1 replicate margin + 10% buffer
    sg_full = per_run_sg * mult
    hdr_full = per_run_hdr * mult
    sgRNA_full_label.value = f"<b>sgRNA per run:</b> {per_run_sg:.2f} µg &nbsp;&nbsp; <b>full screen (×{mult:.2f}):</b> {sg_full:.2f} µg"
    HDR_full_label.value = f"<b>HDR per run:</b> {per_run_hdr:.2f} µg &nbsp;&nbsp; <b>full screen (×{mult:.2f}):</b> {hdr_full:.2f} µg"

# attach updater observers
sgRNA_s.observe(update_full_screen_labels, 'value')
HDR_s.observe(update_full_screen_labels, 'value')
cells_transfected_s.observe(update_full_screen_labels, 'value')
update_full_screen_labels() # Initial call

# ------------------------------------------------------
# Output areas for main display
# ------------------------------------------------------
output_text = widgets.Output()
coverage_plot_out = widgets.Output()

# ------------------------------------------------------
# Core model computation
# ------------------------------------------------------
def compute_metrics(cells_transfected, editing_eff, library_size, cells_retained, cells_pelleted, genomes_input, reads_per_replicate):
    # quantities (absolute)
    selected_cells = cells_transfected # All surviving are transfected after selection
    edited_cells = selected_cells * editing_eff
    edited_cells_retained = edited_cells * (cells_retained / selected_cells) if selected_cells != 0 else 0.0
    # edited genomes entering PCR estimated by scaling genomes_input by fraction of edited cells in pelleted vs pelleted total
    # Assumption: edited cells distribute proportionally between retained and pelleted.
    edited_genomes_PCR = genomes_input * (edited_cells_retained / cells_pelleted) if cells_pelleted != 0 else 0.0

    # Coverage metrics
    genomes_per_design = genomes_input / library_size if library_size != 0 else 0.0
    edited_genomes_per_design = genomes_per_design * editing_eff
    reads_per_design = reads_per_replicate / library_size if library_size != 0 else 0.0
    reads_per_edited_genome = reads_per_design / edited_genomes_per_design if edited_genomes_per_design != 0 else np.nan

    # coverage stages per design (for Matplotlib plot)
    coverage_stages = {
        "Transfected": selected_cells / library_size if library_size != 0 else 0.0,
        "Edited": edited_cells / library_size if library_size != 0 else 0.0,
        "Edited Retained": edited_cells_retained / library_size if library_size != 0 else 0.0,
        "PCR Input": genomes_per_design,
        "Reads": reads_per_design
    }

    return dict(
        selected_cells=selected_cells,
        edited_cells=edited_cells,
        edited_cells_retained=edited_cells_retained,
        edited_genomes_PCR=edited_genomes_PCR,
        genomes_per_design=genomes_per_design,
        edited_genomes_per_design=edited_genomes_per_design,
        reads_per_design=reads_per_design,
        reads_per_edited_genome=reads_per_edited_genome,
        coverage_stages=coverage_stages
    )

# ------------------------------------------------------
# Main output display function (text + matplotlib coverage)
# ------------------------------------------------------
def render_main_output(
    DNA_sgRNA_per_million,
    DNA_HDR_per_million,
    cells_transfected,
    transfection_eff,
    editing_eff,
    library_size,
    cells_retained,
    cells_pelleted,
    genomes_input,
    reads_per_replicate
):
    with output_text:
        clear_output(wait=True)
        # DNA totals
        DNA_sgRNA_total = DNA_sgRNA_per_million * (cells_transfected / 1e6)
        DNA_HDR_total = DNA_HDR_per_million * (cells_transfected / 1e6)
        mult = 3.1 * 1.1 # Multiplier for full screen calculation
        DNA_sgRNA_full = DNA_sgRNA_total * mult
        DNA_HDR_full = DNA_HDR_total * mult

        # compute metrics
        m = compute_metrics(cells_transfected, editing_eff, library_size, cells_retained, cells_pelleted, genomes_input, reads_per_replicate)

        # textual report
        print("=== Transfection Inputs ===")
        print(f"Cells transfected: {cells_transfected/1e6:.1f} M")
        print(f"sgRNA DNA: {DNA_sgRNA_total:.2f} µg total ({DNA_sgRNA_per_million:.3f} µg / 10⁶ cells)")
        print(f"HDR DNA:   {DNA_HDR_total:.2f} µg total ({DNA_HDR_per_million:.3f} µg / 10⁶ cells)\n")
        print(f"Full screen requirement (3× reps + 0.1 margin + 10% buffer): ×{mult:.2f}")
        print(f"→ sgRNA total required: {DNA_sgRNA_full:.2f} µg")
        print(f"→ HDR total required:   {DNA_HDR_full:.2f} µg\n")

        print("=== Post-Selection Outcomes ===")
        print(f"Editing efficiency: {editing_eff*100:.0f}%")
        print(f"Edited cells retained: {m['edited_cells_retained']/1e6:.2f} M")
        print(f"Edited genomes entering PCR: {m['edited_genomes_PCR']/1e6:.2f} M\n")

        print("=== Coverage Metrics ===")
        print(f"Total genomes per design (PCR input): {m['genomes_per_design']:.1f}")
        print(f"Edited genomes per design: {m['edited_genomes_per_design']:.1f}")
        print(f"Edited retained per design: {m['coverage_stages']['Edited Retained']:.1f}")
        print(f"Total reads per design: {m['reads_per_design']:.1f}")
        rpmg = m['reads_per_edited_genome']
        if np.isnan(rpmg):
            print(f"Reads per edited genome: n/a (no edited genomes present)")
        else:
            print(f"Reads per edited genome: {rpmg:.2f}\n")

    # render coverage plot with annotated values using Matplotlib
    with coverage_plot_out:
        clear_output(wait=True)
        stages = list(m['coverage_stages'].keys())
        values = list(m['coverage_stages'].values())
        plt.figure(figsize=(7,4.5))
        plt.plot(stages, values, marker='o', lw=2)
        plt.yscale('log')
        plt.ylabel('Coverage per variant (log scale)')
        plt.title('Coverage Across Pipeline Stages')
        plt.grid(True, which="both", ls="--", lw=0.5)

        # annotate points with numeric values (compact formatting)
        for i, (stage, val) in enumerate(zip(stages, values)):
            if not np.isfinite(val) or val == 0:
                label = "n/a"
            elif val >= 1000:
                label = f"{val:,.0f}"
            elif val >= 1:
                label = f"{val:.1f}"
            else:
                label = f"{val:.2g}"
            plt.text(i, val * 1.05, label, ha='center', va='bottom', fontsize=9)

        plt.show()

# Connect main output to sliders using explicit observers
controls_map = {
    'DNA_sgRNA_per_million': sgRNA_s,
    'DNA_HDR_per_million': HDR_s,
    'cells_transfected': cells_transfected_s,
    'transfection_eff': transfection_eff_s,
    'editing_eff': editing_eff_s,
    'library_size': library_size_s,
    'cells_retained': cells_retained_s,
    'cells_pelleted': cells_pelleted_s,
    'genomes_input': genomes_input_s,
    'reads_per_replicate': reads_per_replicate_s
}

def update_main_output_on_change(change):
    render_main_output(
        DNA_sgRNA_per_million=sgRNA_s.value,
        DNA_HDR_per_million=HDR_s.value,
        cells_transfected=cells_transfected_s.value,
        transfection_eff=transfection_eff_s.value,
        editing_eff=editing_eff_s.value,
        library_size=library_size_s.value,
        cells_retained=cells_retained_s.value,
        cells_pelleted=cells_pelleted_s.value,
        genomes_input=genomes_input_s.value,
        reads_per_replicate=reads_per_replicate_s.value
    )

for param_name, slider_widget in controls_map.items():
    slider_widget.observe(update_main_output_on_change, names='value')

# Initial render for the main output
update_main_output_on_change(None) # Call once initially with a dummy change object

# =============================================
# Sweep Plot (Plotly) with Export Functionality
# =============================================
output_export = widgets.Output(layout=widgets.Layout(border='1px solid #ccc'))

param_options = {
    'editing_eff': ('Editing efficiency', editing_eff_s),
    'cells_transfected': ('Cells transfected', cells_transfected_s),
    'genomes_input': ('Genomes into PCR', genomes_input_s),
    'reads_per_replicate': ('Reads per replicate', reads_per_replicate_s),
    'library_size': ('Library size', library_size_s)
}

sweep_param_dd = widgets.Dropdown(
    options=[(v[0], k) for k,v in param_options.items()],
    value='editing_eff',
    description='Sweep parameter:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
sweep_points = widgets.IntSlider(value=50, min=10, max=200, step=10, description='Points')
yscale_dd = widgets.Dropdown(options=['linear', 'log'], value='linear', description='Y-scale')
export_button = widgets.Button(description='Generate & Export Sweep', button_style='success')

def run_sweep_export(b):
    with output_export:
        clear_output(wait=True)
        param_key = sweep_param_dd.value
        param_label, slider_obj = param_options[param_key]

        x_vals = np.linspace(slider_obj.min, slider_obj.max, sweep_points.value)

        edited_genomes = []
        reads_per_edited = []
        reads_per_design = []

        # Store fixed parameter values for display
        fixed_params_display = {
            'sgRNA_per_million': sgRNA_s.value,
            'HDR_per_million': HDR_s.value,
            'cells_transfected': cells_transfected_s.value,
            'transfection_eff': transfection_eff_s.value,
            'editing_eff': editing_eff_s.value,
            'library_size': library_size_s.value,
            'cells_retained': cells_retained_s.value,
            'cells_pelleted': cells_pelleted_s.value,
            'genomes_input': genomes_input_s.value,
            'reads_per_replicate': reads_per_replicate_s.value
        }

        for val in x_vals:
            # Construct parameters for compute_metrics, overriding the swept parameter
            metrics_args = {
                'cells_transfected': cells_transfected_s.value,
                'editing_eff': editing_eff_s.value,
                'library_size': library_size_s.value,
                'cells_retained': cells_retained_s.value,
                'cells_pelleted': cells_pelleted_s.value,
                'genomes_input': genomes_input_s.value,
                'reads_per_replicate': reads_per_replicate_s.value
            }
            metrics_args[param_key] = val

            m = compute_metrics(**metrics_args)

            edited_genomes.append(m['edited_genomes_per_design'])
            reads_per_edited.append(m['reads_per_edited_genome'])
            reads_per_design.append(m['reads_per_design'])

        # --- Build Plotly figure ---
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x_vals, y=edited_genomes, mode='lines+markers', name='Edited genomes per design', line=dict(color='royalblue')))
        fig.add_trace(go.Scatter(x=x_vals, y=reads_per_edited, mode='lines+markers', name='Reads per edited genome', yaxis='y2', line=dict(color='darkorange')))
        fig.add_trace(go.Scatter(x=x_vals, y=reads_per_design, mode='lines', name='Reads per design (overall)', line=dict(color='gray', dash='dot')))

        # Add vertical line showing current value
        curr_val = slider_obj.value
        fig.add_vline(x=curr_val, line=dict(color='red', dash='dot'), annotation_text=f'Current: {curr_val:.2g}', annotation_position='top right')

        fig.update_layout(
            title=f"Sweep: {param_label}",
            xaxis_title=param_label,
            yaxis_title='Genomes / Reads per design (left)',
            yaxis=dict(type=yscale_dd.value),
            yaxis2=dict(title='Reads per edited genome (right)', overlaying='y', side='right', type=yscale_dd.value),
            template='plotly_white',
            legend=dict(x=0.01, y=0.99),
            width=900,
            height=500,
            margin=dict(r=50)
        )

        # Display fixed parameters and assumptions below the plot
        print(f"<b>Fixed Parameters (current values):</b>")
        for key, value in fixed_params_display.items():
            if key not in ['transfection_eff', param_key]: # transfection_eff is reference, param_key is swept
                formatted_value = f"{value:.2e}" if isinstance(value, (float, np.float64)) and value >= 1e5 else f"{value:.3g}"
                print(f"- {key.replace('_',' ').title()}: {formatted_value}")
        print("<br><b>Assumptions:</b>")
        print("- All surviving cells are transfected after selection.")
        print("- Edited genomes entering PCR scale with edited cells in pelleted sample.")
        print("- Full screen requirement includes 3 replicates + 0.1 margin + 10% buffer.")

        # Save and offer download
        fn = 'sweep_plot.html'
        fig.write_html(fn, include_plotlyjs='cdn')
        print(f"✅ Sweep plot generated and saved as '{fn}'.")
        files.download(fn)

        # Display the plot within the output widget
        display(fig)

export_button.on_click(run_sweep_export)

sweep_tab_content = widgets.VBox([
    widgets.HBox([sweep_param_dd, sweep_points, yscale_dd]),
    export_button,
    output_export
])

# =============================================
# Scenario Comparison Table
# =============================================
output_table = widgets.Output()

def scenario_row_defaults():
    return {
        'cells_transfected': cells_transfected_s.value,
        'editing_eff': editing_eff_s.value,
        'library_size': library_size_s.value,
        'genomes_input': genomes_input_s.value,
        'reads_per_replicate': reads_per_replicate_s.value
    }

def make_scenario_widget(prefix, defaults):
    w_cells = widgets.FloatText(value=defaults['cells_transfected'], description='Cells', layout=widgets.Layout(width='240px'))
    w_edit = widgets.FloatText(value=defaults['editing_eff'], description='Edit eff', layout=widgets.Layout(width='240px'))
    w_lib  = widgets.IntText(value=int(defaults['library_size']), description='Library', layout=widgets.Layout(width='240px'))
    w_genomes = widgets.FloatText(value=defaults['genomes_input'], description='Genomes in PCR', layout=widgets.Layout(width='240px'))
    w_reads = widgets.FloatText(value=defaults['reads_per_replicate'], description='Reads', layout=widgets.Layout(width='240px'))
    box = widgets.VBox([widgets.HTML(f"<b>{prefix}</b>"), w_cells, w_edit, w_lib, w_genomes, w_reads])
    return {
        'box': box,
        'cells': w_cells,
        'edit': w_edit,
        'lib': w_lib,
        'genomes': w_genomes,
        'reads': w_reads
    }

scenario1 = make_scenario_widget('Scenario A', scenario_row_defaults())
scenario2 = make_scenario_widget('Scenario B', scenario_row_defaults())
scenario3 = make_scenario_widget('Scenario C', scenario_row_defaults())

compare_button = widgets.Button(description='Compare scenarios', button_style='success')

def compare_scenarios(b):
    with output_table:
        clear_output(wait=True)
        scenarios = [scenario1, scenario2, scenario3]
        cols = []
        for i, s in enumerate(scenarios):
            params = {
                'cells_transfected': float(s['cells'].value),
                'editing_eff': float(s['edit'].value),
                'library_size': float(s['lib'].value),
                'cells_retained': float(cells_retained_s.value), # Fixed from main sliders
                'cells_pelleted': float(cells_pelleted_s.value), # Fixed from main sliders
                'genomes_input': float(s['genomes'].value),
                'reads_per_replicate': float(s['reads'].value)
            }
            m = compute_metrics(params['cells_transfected'], params['editing_eff'], params['library_size'],
                                params['cells_retained'], params['cells_pelleted'], params['genomes_input'],
                                params['reads_per_replicate'])
            col = {
                'Edited cells retained (M)': m['edited_cells_retained']/1e6,
                'Edited genomes entering PCR (M)': m['edited_genomes_PCR']/1e6,
                'Genomes per design': m['genomes_per_design'],
                'Edited genomes per design': m['edited_genomes_per_design'],
                'Reads per design': m['reads_per_design'],
                'Reads per edited genome': m['reads_per_edited_genome']
            }
            cols.append(col)
            cols[-1]['Scenario'] = f"Scenario {chr(ord('A')+i)}"
        # DataFrame
        df = pd.DataFrame(cols).set_index('Scenario').T
        display(df.style.format("{:.3g}"))

        # Bar chart: Selected metrics comparison
        fig = go.Figure()
        metrics_to_plot = ['Edited genomes per design', 'Reads per design', 'Reads per edited genome']

        # Prepare data for grouped bar chart
        scenario_names = [f"Scenario {chr(ord('A')+i)}" for i in range(len(scenarios))]
        data = []
        for metric in metrics_to_plot:
            for i, scenario_name in enumerate(scenario_names):
                bar_data = {
                    'x': [metric],
                    'y': [df.loc[metric, scenario_name]],
                    'name': f'{scenario_name} - {metric}',
                    'marker_color': px.colors.qualitative.Plotly[i] # Use Plotly default colors
                }
                data.append(go.Bar(bar_data))

        fig.update_layout(
            barmode='group',
            title='Scenario metrics (per-design values)',
            xaxis_title="Metric",
            yaxis_title="Value",
            legend_title="Scenario / Metric",
            template="plotly_white"
        )
        fig.add_traces(data)
        fig.show()

compare_button.on_click(compare_scenarios)

scenarios_box = widgets.HBox([scenario1['box'], scenario2['box'], scenario3['box']])
scenario_tab_content = widgets.VBox([scenarios_box, compare_button, output_table])

# ------------------------------------------------------
# Tabbed Interface for advanced visualizations
# ------------------------------------------------------
tabs = widgets.Tab(children=[sweep_tab_content, scenario_tab_content])
tabs.set_title(0, 'Sweep Plot')
tabs.set_title(1, 'Scenario Table')

# ------------------------------------------------------
# Final Layout Assembly
# ------------------------------------------------------
full_ui_layout = widgets.VBox([
    widgets.HTML("<h2>SGE Screen Interactive Model</h2>"),
    widgets.HTML("<h3>DNA Inputs</h3>"),
    sgRNA_box,
    HDR_box,
    ratio_box,
    widgets.HBox([sgRNA_full_label, HDR_full_label]),
    widgets.HTML("<h3>Cell & Editing Parameters</h3>"),
    cells_transfected_box,
    transfection_eff_box,
    editing_eff_box,
    library_size_box,
    cells_retained_box,
    cells_pelleted_box,
    genomes_input_box,
    reads_per_replicate_box,
    widgets.HTML("<hr><h3>Model Output</h3>"),
    output_text,
    coverage_plot_out,
    widgets.HTML("<hr><h3>Advanced Visualizations</h3>"),
    tabs
])

display(full_ui_layout)


# Section 11 — Mechanistic simulation (library skew + stochastic sampling)

The original toy model above treats coverage as a **single mean value per design** (e.g. `cells_transfected / library_size`).

This section adds what was discussed in the *Simulation model* notes:
- **Coverage is a distribution**, not just a mean (because the library can be *skewed*).
- A tunable **library skew** parameter (log-normal or Dirichlet) controls how unevenly variants are represented.
- A tunable **HDR / editing rate** controls how many cells actually integrate the template.
- Stochastic sampling is applied at each stage (retain → pellet → genomes into PCR → reads), so we can predict **dropouts** and **quantiles** (e.g., 10th percentile coverage).

In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# Utility stats
# -----------------------------
def gini_coefficient(x: np.ndarray) -> float:
    """
    Gini coefficient (0 = perfectly even, 1 = maximally skewed).
    Works for nonnegative vectors. If all zeros, returns 0.
    """
    x = np.asarray(x, dtype=float)
    x = x[x >= 0]
    if x.size == 0:
        return 0.0
    s = x.sum()
    if s == 0:
        return 0.0
    x = np.sort(x)
    n = x.size
    # Gini = (2*sum(i*x_i)/(n*sum(x))) - (n+1)/n
    i = np.arange(1, n + 1)
    return float((2.0 * np.sum(i * x) / (n * s)) - (n + 1) / n)

def summarize_counts(counts: np.ndarray, name: str = "counts") -> dict:
    counts = np.asarray(counts, dtype=float)
    out = {
        f"{name}_mean": float(np.mean(counts)),
        f"{name}_median": float(np.median(counts)),
        f"{name}_p10": float(np.quantile(counts, 0.10)),
        f"{name}_p01": float(np.quantile(counts, 0.01)),
        f"{name}_min": float(np.min(counts)),
        f"{name}_max": float(np.max(counts)),
        f"{name}_cv": float(np.std(counts) / np.mean(counts)) if np.mean(counts) > 0 else np.nan,
        f"{name}_gini": gini_coefficient(counts),
        f"{name}_dropout_frac": float(np.mean(counts <= 0)),
    }
    return out

# -----------------------------
# Library skew models
# -----------------------------
def library_fractions(
    library_size: int,
    skew_model: str = "lognormal",
    skew_param: float = 0.7,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """
    Returns a probability vector p_i (sums to 1) describing how abundant each library element is.

    skew_model:
      - "lognormal": skew_param = sigma of log-abundance (0 => uniform)
      - "dirichlet": skew_param = concentration alpha (small => skewed, large => uniform)
    """
    if rng is None:
        rng = np.random.default_rng()
    L = int(library_size)
    if L <= 0:
        raise ValueError("library_size must be > 0")

    skew_model = skew_model.lower().strip()
    if skew_model == "lognormal":
        sigma = float(skew_param)
        if sigma < 0:
            raise ValueError("lognormal sigma must be >= 0")
        # log-abundances; mean doesn't matter because we normalize
        log_a = rng.normal(loc=0.0, scale=sigma, size=L)
        a = np.exp(log_a)
        p = a / a.sum()
        return p

    if skew_model == "dirichlet":
        alpha = float(skew_param)
        if alpha <= 0:
            raise ValueError("dirichlet alpha must be > 0")
        p = rng.dirichlet(alpha * np.ones(L))
        return p

    raise ValueError("skew_model must be one of: 'lognormal', 'dirichlet'")



def estimate_skew_from_counts(
    counts: np.ndarray,
    skew_model: str = "lognormal",
    n_mc: int = 200,
    random_state: int = 0,
) -> float:
    """
    Roughly estimate the skew parameter from an observed per-design count vector.

    Approach:
      1) compute observed Gini(counts)
      2) find skew_param whose expected Gini(p) matches, using binary search + Monte Carlo.

    Useful when you have (e.g.) a plasmid-seq distribution or an early timepoint read distribution
    and want to set `skew_param` to something realistic.
    """
    counts = np.asarray(counts, dtype=float)
    if counts.ndim != 1:
        raise ValueError("counts must be 1D")
    if np.any(counts < 0):
        raise ValueError("counts must be nonnegative")

    L = int(counts.size)
    target = gini_coefficient(counts)

    rng = np.random.default_rng(random_state)

    def expected_gini(param: float) -> float:
        gs = []
        for _ in range(int(n_mc)):
            p = library_fractions(L, skew_model=skew_model, skew_param=param, rng=rng)
            gs.append(gini_coefficient(p))
        return float(np.mean(gs))

    skew_model_l = skew_model.lower().strip()

    if skew_model_l == "lognormal":
        lo, hi = 0.0, 3.0
        g_lo, g_hi = expected_gini(lo), expected_gini(hi)
        if target <= g_lo:
            return lo
        if target >= g_hi:
            return hi
        for _ in range(25):
            mid = 0.5 * (lo + hi)
            if expected_gini(mid) < target:
                lo = mid
            else:
                hi = mid
        return 0.5 * (lo + hi)

    if skew_model_l == "dirichlet":
        lo, hi = 1e-3, 200.0
        g_lo, g_hi = expected_gini(lo), expected_gini(hi)
        if target >= g_lo:
            return lo
        if target <= g_hi:
            return hi
        for _ in range(25):
            mid = 0.5 * (lo + hi)
            if expected_gini(mid) > target:
                lo = mid
            else:
                hi = mid
        return 0.5 * (lo + hi)

    raise ValueError("skew_model must be one of: 'lognormal', 'dirichlet'")


# -----------------------------
# Mechanistic pipeline simulation
# -----------------------------
def simulate_once(
    *,
    library_size: int = 2500,
    cells_transfected: int = int(20e6),
    hdr_rate: float = 0.4,                 # fraction of cells that successfully integrate (HDR / editing)
    cells_retained: int = int(5e6),
    cells_pelleted: int = int(3e6),
    genomes_input: int = int(1.52e6),      # genome equivalents into PCR
    reads_per_replicate: int = int(10e6),
    skew_model: str = "lognormal",
    skew_param: float = 0.7,
    design_edit_kappa: float = 80.0,       # higher => less design-to-design editing variability
    assignment_model: str = "poisson",     # "poisson" (fast) or "multinomial" (exact total)
    rng: np.random.Generator | None = None,
) -> dict:
    """
    Simulate one replicate of the screen at the 'per-design' level.
    Returns arrays and summary metrics.

    Key outputs:
      - cell_counts: cells per design after selection / transfection
      - edited_counts: edited cells per design
      - retained_edited: edited cells per design in retained pool
      - pcr_edited: edited genomes per design entering PCR
      - reads: reads per design for edited alleles
    """
    if rng is None:
        rng = np.random.default_rng()

    L = int(library_size)
    N = int(cells_transfected)
    N_ret = int(cells_retained)
    N_pel = int(cells_pelleted)
    G_in = int(genomes_input)
    R = int(reads_per_replicate)

    if not (0 <= hdr_rate <= 1):
        raise ValueError("hdr_rate must be in [0, 1]")
    if N <= 0 or L <= 0:
        raise ValueError("cells_transfected and library_size must be > 0")

    # 1) Library abundance / skew
    p = library_fractions(L, skew_model=skew_model, skew_param=skew_param, rng=rng)

    # 2) Allocate cells to designs
    assignment_model = assignment_model.lower().strip()
    if assignment_model == "multinomial":
        cell_counts = rng.multinomial(N, pvals=p)
    elif assignment_model == "poisson":
        # Fast approximation; total varies ~Poisson(N).
        cell_counts = rng.poisson(lam=N * p)
    else:
        raise ValueError("assignment_model must be 'poisson' or 'multinomial'")

    # 3) Design-to-design editing variability (optional)
    #    p_edit_i ~ Beta(a,b) with mean hdr_rate and concentration design_edit_kappa
    kappa = float(design_edit_kappa)
    kappa = max(kappa, 1e-6)
    a = max(hdr_rate * kappa, 1e-6)
    b = max((1 - hdr_rate) * kappa, 1e-6)
    p_edit_i = rng.beta(a, b, size=L)
    edited_counts = rng.binomial(cell_counts, p=p_edit_i)

    # 4) Retained pool (sampling fraction)
    #    Approximate as independent binomial thinning.
    if N_ret <= 0:
        retained_edited = np.zeros(L, dtype=int)
    else:
        retain_frac = min(max(N_ret / max(N, 1), 0.0), 1.0)
        retained_edited = rng.binomial(edited_counts, p=retain_frac)

    # 5) Pelleted pool (sampling fraction)
    #    We need *pelleted total per design* and *pelleted edited per design* for PCR sampling.
    if N_pel <= 0:
        pelleted_counts = np.zeros(L, dtype=int)
        pelleted_edited = np.zeros(L, dtype=int)
    else:
        pel_frac = min(max(N_pel / max(N, 1), 0.0), 1.0)
        pelleted_counts = rng.binomial(cell_counts, p=pel_frac)
        pelleted_edited = rng.binomial(edited_counts, p=pel_frac)

    # 6) Genomes into PCR: sample genomes from pelleted pool,
    #    then count how many of those are edited (binomial approx).
    if G_in <= 0 or pelleted_counts.sum() <= 0:
        pcr_counts = np.zeros(L, dtype=int)
        pcr_edited = np.zeros(L, dtype=int)
    else:
        # sample genomes across designs proportionally
        probs = pelleted_counts / pelleted_counts.sum()
        pcr_counts = rng.multinomial(G_in, pvals=probs)

        # fraction edited per design in pelleted pool
        frac_edited = np.divide(
            pelleted_edited,
            pelleted_counts,
            out=np.zeros_like(pelleted_edited, dtype=float),
            where=pelleted_counts > 0,
        )
        pcr_edited = rng.binomial(pcr_counts, p=frac_edited)

    # 7) Sequencing reads: allocate reads across edited genomes
    if R <= 0 or pcr_edited.sum() <= 0:
        reads = np.zeros(L, dtype=int)
    else:
        probs = pcr_edited / pcr_edited.sum()
        reads = rng.multinomial(R, pvals=probs)

    # Summary metrics at each stage
    metrics = {}
    metrics.update(summarize_counts(cell_counts, "cells"))
    metrics.update(summarize_counts(edited_counts, "edited"))
    metrics.update(summarize_counts(retained_edited, "edited_retained"))
    metrics.update(summarize_counts(pcr_edited, "edited_pcr"))
    metrics.update(summarize_counts(reads, "reads"))

    # Global / convenience metrics
    metrics["library_size"] = L
    metrics["cells_transfected_total"] = int(cell_counts.sum())
    metrics["reads_total"] = int(reads.sum())
    metrics["skew_model"] = skew_model
    metrics["skew_param"] = float(skew_param)
    metrics["plasmid_gini"] = gini_coefficient(p)
    metrics["plasmid_p01"] = float(np.quantile(p, 0.01))
    metrics["plasmid_p10"] = float(np.quantile(p, 0.10))
    metrics["plasmid_max"] = float(np.max(p))

    return {
        "p": p,
        "cell_counts": cell_counts,
        "edited_counts": edited_counts,
        "retained_edited": retained_edited,
        "pcr_edited": pcr_edited,
        "reads": reads,
        "metrics": metrics,
    }

def run_monte_carlo(n_reps: int = 50, random_state: int = 0, **params) -> pd.DataFrame:
    """
    Run many stochastic replicates and return a DataFrame of per-replicate summary metrics.
    """
    rng = np.random.default_rng(random_state)
    rows = []
    for _ in range(int(n_reps)):
        sim = simulate_once(rng=rng, **params)
        rows.append(sim["metrics"])
    return pd.DataFrame(rows)

# -----------------------------
# Example: run once and inspect key outputs
# -----------------------------
example = simulate_once(
    library_size=2500,
    cells_transfected=int(20e6),
    hdr_rate=0.4,
    cells_retained=int(5e6),
    cells_pelleted=int(3e6),
    genomes_input=int(1.52e6),
    reads_per_replicate=int(10e6),
    skew_model="lognormal",
    skew_param=0.7,
    assignment_model="poisson",
    design_edit_kappa=80.0,
    rng=np.random.default_rng(1),
)

example["metrics"]

# Section 12 — Synthetic data + interpretable statistical “equation”

Below is a simple workflow to:
1) Generate synthetic training data by sampling plausible experimental parameters and simulating outcomes.
2) Fit an interpretable regression model that approximates a mapping:

**experimental parameters → outcome metric**

This gives you a *statistical equation* you can use for quick prediction and for suggesting new experimental setups.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
import numpy as np
import pandas as pd

def sample_parameters(n: int, rng: np.random.Generator | None = None) -> pd.DataFrame:
    """
    Sample a plausible parameter space. Adjust ranges as you learn more.
    """
    if rng is None:
        rng = np.random.default_rng()

    # Note: coverage_mean ≈ cells_transfected / library_size
    library_size = rng.integers(500, 5001, size=n)

    # Sample a target mean coverage and convert to cells_transfected
    coverage_mean = rng.uniform(100, 1000, size=n)  # 100x–1000x
    cells_transfected = np.round(coverage_mean * library_size).astype(int)

    hdr_rate = rng.uniform(0.05, 0.8, size=n)  # 5%–80%
    skew_param = rng.uniform(0.0, 1.5, size=n) # lognormal sigma (0 uniform)

    # Downstream sampling
    cells_retained = np.round(cells_transfected * rng.uniform(0.15, 0.40, size=n)).astype(int)
    cells_pelleted = np.round(cells_transfected * rng.uniform(0.10, 0.30, size=n)).astype(int)

    genomes_input = rng.integers(int(0.5e6), int(5e6)+1, size=n)
    reads_per_replicate = rng.integers(int(1e6), int(20e6)+1, size=n)

    # Design-level edit variability knob
    design_edit_kappa = rng.uniform(30, 200, size=n)

    # Optional: "GC content" placeholder feature (mean GC for the library)
    gc_mean = rng.uniform(0.35, 0.65, size=n)

    return pd.DataFrame({
        "library_size": library_size,
        "coverage_mean": coverage_mean,
        "cells_transfected": cells_transfected,
        "cells_retained": cells_retained,
        "cells_pelleted": cells_pelleted,
        "hdr_rate": hdr_rate,
        "skew_param": skew_param,
        "genomes_input": genomes_input,
        "reads_per_replicate": reads_per_replicate,
        "design_edit_kappa": design_edit_kappa,
        "gc_mean": gc_mean,
    })

def build_synthetic_dataset(
    n_samples: int = 1000,
    n_reps_per_sample: int = 10,
    random_state: int = 0,
    skew_model: str = "lognormal",
    assignment_model: str = "poisson",
) -> pd.DataFrame:
    """
    Generate synthetic data by sampling parameters and running Monte-Carlo simulation.

    Returns one row per sampled parameter set, with mean outcome metrics across replicates.
    """
    rng = np.random.default_rng(random_state)
    params_df = sample_parameters(n_samples, rng=rng)

    rows = []
    for _, row in params_df.iterrows():
        # Run Monte Carlo replicates for this parameter set
        mc = run_monte_carlo(
            n_reps=n_reps_per_sample,
            random_state=int(rng.integers(0, 1_000_000)),
            library_size=int(row.library_size),
            cells_transfected=int(row.cells_transfected),
            hdr_rate=float(row.hdr_rate),
            cells_retained=int(row.cells_retained),
            cells_pelleted=int(row.cells_pelleted),
            genomes_input=int(row.genomes_input),
            reads_per_replicate=int(row.reads_per_replicate),
            skew_model=skew_model,
            skew_param=float(row.skew_param),
            design_edit_kappa=float(row.design_edit_kappa),
            assignment_model=assignment_model,
        )
        # Aggregate MC outcomes (mean across replicates)
        out = row.to_dict()
        for col in ["reads_dropout_frac", "edited_retained_p10", "reads_p10", "reads_gini", "plasmid_gini"]:
            out[col] = float(mc[col].mean())
        rows.append(out)

    return pd.DataFrame(rows)

def fit_interpretable_equation(df: pd.DataFrame, target_col: str):
    """
    Fit a simple ridge regression on a standardized feature set.

    The output is an explicit linear equation (in standardized feature space).
    """
    # Features
    feature_cols = [
        "library_size",
        "coverage_mean",
        "hdr_rate",
        "skew_param",
        "genomes_input",
        "reads_per_replicate",
        "design_edit_kappa",
        "gc_mean",
    ]
    X = df[feature_cols].copy()

    # Some targets are bounded in [0,1]. Use logit transform for stability.
    y = df[target_col].values.astype(float)
    y_is_logit = False
    if target_col.endswith("_dropout_frac"):
        eps = 1e-6
        y = np.log((y + eps) / (1 - y + eps))  # logit
        y_is_logit = True

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
    ])
    model.fit(X, y)

    # Build a readable equation string
    scaler: StandardScaler = model.named_steps["scaler"]
    ridge: Ridge = model.named_steps["ridge"]

    coefs = ridge.coef_
    intercept = ridge.intercept_

    terms = []
    for name, coef, mean, scale in zip(feature_cols, coefs, scaler.mean_, scaler.scale_):
        terms.append(f"({coef:+.4f}) * (({name} - {mean:.4g}) / {scale:.4g})")

    equation = "y_hat = " + f"{intercept:.4f} " + " ".join(terms)

    if y_is_logit:
        equation += "\n(note: y_hat is logit(dropout); convert via sigmoid: p = 1/(1+exp(-y_hat)))"

    return model, feature_cols, equation

# -----------------------------
# Example: build a small dataset and fit a model
# -----------------------------
# (Increase n_samples/n_reps_per_sample later; start small to validate plumbing.)
syn_df = build_synthetic_dataset(n_samples=200, n_reps_per_sample=5, random_state=1)
model, features, equation = fit_interpretable_equation(syn_df, target_col="reads_dropout_frac")

equation[:400] + " ..."

# Section 13 — Suggest experimental parameters for a desired outcome (“inverse design”)

Given a desired outcome (e.g. **<1% dropout at the read stage** and **≥X edited cells per design at the 10th percentile**),
we can search the parameter space for candidate experimental conditions.

Below is a two-stage method:
1) **Fast screen** with the fitted statistical model.
2) **Re-score** the top candidates using the full mechanistic simulation.

In [ ]:
import numpy as np
import pandas as pd

def suggest_experiments(
    surrogate_model,
    feature_cols,
    target_constraints: dict,
    n_candidates: int = 5000,
    top_k: int = 15,
    random_state: int = 0,
    n_reps_rescore: int = 25,
):
    """
    target_constraints keys can include:
      - "reads_dropout_frac_max" (e.g. 0.01)
      - "edited_retained_p10_min" (e.g. 200)
      - "reads_p10_min" (e.g. 50)

    This function:
      - samples n_candidates parameter sets
      - ranks by surrogate predictions (lower dropout is better)
      - keeps top_k
      - re-simulates them with Monte Carlo and returns a ranked DataFrame
    """
    rng = np.random.default_rng(random_state)
    cand = sample_parameters(n_candidates, rng=rng)

    # Surrogate predicts logit(dropout) if that is what we trained it on.
    y_hat = surrogate_model.predict(cand[feature_cols])

    # For dropout we want low, so we rank by predicted y_hat (lower logit => lower dropout)
    cand = cand.copy()
    cand["surrogate_score"] = y_hat

    # Keep top-K lowest surrogate score
    top = cand.nsmallest(top_k, "surrogate_score").reset_index(drop=True)

    # Re-score with mechanistic simulation
    rescored_rows = []
    for _, row in top.iterrows():
        mc = run_monte_carlo(
            n_reps=n_reps_rescore,
            random_state=int(rng.integers(0, 1_000_000)),
            library_size=int(row.library_size),
            cells_transfected=int(row.cells_transfected),
            hdr_rate=float(row.hdr_rate),
            cells_retained=int(row.cells_retained),
            cells_pelleted=int(row.cells_pelleted),
            genomes_input=int(row.genomes_input),
            reads_per_replicate=int(row.reads_per_replicate),
            skew_model="lognormal",
            skew_param=float(row.skew_param),
            design_edit_kappa=float(row.design_edit_kappa),
            assignment_model="poisson",
        )
        out = row.to_dict()
        out["reads_dropout_frac_mean"] = float(mc["reads_dropout_frac"].mean())
        out["edited_retained_p10_mean"] = float(mc["edited_retained_p10"].mean())
        out["reads_p10_mean"] = float(mc["reads_p10"].mean())
        out["reads_gini_mean"] = float(mc["reads_gini"].mean())
        rescored_rows.append(out)

    res = pd.DataFrame(rescored_rows)

    # Apply constraints and rank
    mask = np.ones(len(res), dtype=bool)
    if "reads_dropout_frac_max" in target_constraints:
        mask &= res["reads_dropout_frac_mean"] <= target_constraints["reads_dropout_frac_max"]
    if "edited_retained_p10_min" in target_constraints:
        mask &= res["edited_retained_p10_mean"] >= target_constraints["edited_retained_p10_min"]
    if "reads_p10_min" in target_constraints:
        mask &= res["reads_p10_mean"] >= target_constraints["reads_p10_min"]

    filtered = res[mask].copy()
    if len(filtered) == 0:
        # If nothing passes, return the rescored table anyway (best effort)
        return res.sort_values("reads_dropout_frac_mean").reset_index(drop=True)

    return filtered.sort_values("reads_dropout_frac_mean").reset_index(drop=True)

# -----------------------------
# Example: find candidates with <1% dropout and decent low-percentile coverage
# -----------------------------
targets = {
    "reads_dropout_frac_max": 0.01,
    "edited_retained_p10_min": 200,
    "reads_p10_min": 50,
}

recommendations = suggest_experiments(
    surrogate_model=model,
    feature_cols=features,
    target_constraints=targets,
    n_candidates=2000,
    top_k=10,
    random_state=2,
    n_reps_rescore=10,
)

recommendations.head(10)